# Congestion prediction on LargeST-SD (2019)

Staged per the approved proposal. **Every gate ends with a STOP cell — run up to it, report the printout, wait for go-ahead.**

| Stage | Runtime | Cells |
|---|---|---|
| 0 · audit + baselines | CPU | 00–06b ✅ passed |
| 1 · overfit test | T4 ~2 min · M2 ~10 min | 00, 01, then 07–09 ✅ passed (M2) |
| 2 · subset timing | T4 ~10 min · M2 ~40 min | 00, 01, 07, 08, then 10–11 |
| 3 · R0 first 25% (Gate 3) | M2 ~1 h, background | 00, 01, 07, 08, then 12–15 |
| 4 · full runs | M2 overnight, resumable | after Gate 3 |

Conventions: all fitted statistics use timesteps `[0, TRAIN_END)` only (LargeST's scaler window).
Raw flow `F` is veh/5 min (15-min mean, rounded) — LargeST's own processing.
Small artifacts go to `MyDrive/traffic/congestion/`; everything heavy stays in `/content/work`.

In [1]:
# ── 00 · Bootstrap ─────────────────────────────────────────────────────────
# Mount Drive, clone LargeST, restore the SD artifacts, install missing packages,
# fix seeds and paths. Idempotent — re-run first thing in every new session.
# Colab: Drive at /content/drive. Elsewhere (e.g. the M2): project folder = cwd, data in <project>/data;
# CONG_ROOT / CONG_DRIVE override either.
import os, sys, json, shutil, subprocess, random, time, warnings, platform
from pathlib import Path
import numpy as np
import pandas as pd

ON_COLAB   = Path('/content').is_dir()
ROOT       = Path(os.environ.get('CONG_ROOT', '/content' if ON_COLAB else Path.cwd()))
MYDRIVE    = Path(os.environ.get('CONG_DRIVE', ROOT / 'drive' / 'MyDrive' if ON_COLAB else ROOT / 'data'))
DRIVE      = MYDRIVE / 'traffic'
DRIVE_SD   = DRIVE / 'sd_2019'          # existing pipeline outputs (read-only here)
DRIVE_PROJ = DRIVE / 'congestion'       # new small artifacts
WORK       = ROOT / 'work'
REPO       = WORK / 'LargeST'
DATA       = REPO / 'data' / 'sd'
FACT       = WORK / 'factors'           # local mirror of DRIVE_PROJ/factors

KERNEL = f'python={sys.executable} | {platform.platform()} | host={platform.node()}'
if not DRIVE_SD.exists() and ON_COLAB and 'CONG_DRIVE' not in os.environ:
    from google.colab import drive
    drive.mount(str(ROOT / 'drive'))
assert DRIVE_SD.exists(), (f'{DRIVE_SD} not found — ' + ('Drive mount failed?' if ON_COLAB else
                           'unzip the Drive download of traffic/sd_2019 into that folder'))

for d in [WORK, FACT, WORK / 'cong', *(DRIVE_PROJ / s for s in ('factors', 'ckpt', 'logs', 'results'))]:
    d.mkdir(parents=True, exist_ok=True)

if not (REPO / 'src').exists():
    tmp = WORK / '_largest_clone'
    shutil.rmtree(tmp, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/liuxu77/LargeST.git', str(tmp)], check=True)
    shutil.copytree(tmp, REPO, dirs_exist_ok=True)
    shutil.rmtree(tmp)

SD_FILES = ['sd_his_2019.h5', 'sd_meta.csv', 'sd_rn_adj.npy', '2019/his.npz',
            '2019/idx_train.npy', '2019/idx_val.npy', '2019/idx_test.npy']
for f in SD_FILES:
    if not (DATA / f).exists():
        (DATA / f).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(DRIVE_SD / f, DATA / f)
        print('restored', f)

def ensure(module, pip_name=None):
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name or module], check=True)

ensure('tables')      # pandas.read_hdf backend
ensure('holidays')    # used from 4b onwards

SEED = 2023
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
seed_everything()

def save_factor(name, **arrays):
    """Small derived arrays -> <FACT>/<name>.npz, mirrored to Drive."""
    p = FACT / f'{name}.npz'
    np.savez_compressed(p, **arrays)
    shutil.copy2(p, DRIVE_PROJ / 'factors' / p.name)
    return p

def load_factor(name):
    p = FACT / f'{name}.npz'
    if not p.exists():
        shutil.copy2(DRIVE_PROJ / 'factors' / p.name, p)
    with np.load(p, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

def save_result(name, obj):
    with open(DRIVE_PROJ / 'results' / f'{name}.json', 'w') as fh:
        json.dump(obj, fh, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

warnings.filterwarnings('ignore', message='(All-NaN slice|Mean of empty slice)')
T_IN, HORIZON, STEPS_PER_DAY = 12, 12, 96
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True,
                     text=True).stdout.strip() if shutil.which('nvidia-smi') else 'none'
print(f'{KERNEL}\nseed={SEED}  root={ROOT}  drive={MYDRIVE}  gpu={GPU}  numpy={np.__version__}  pandas={pd.__version__}')

python=/Users/lavanyapareek/Documents/Capstone_Post_Joining/.venv/bin/python | macOS-26.2-arm64-arm-64bit-Mach-O | host=Lavanyas-MacBook-Air.local
seed=2023  root=/Users/lavanyapareek/Documents/Capstone_Post_Joining  drive=/Users/lavanyapareek/Documents/Capstone_Post_Joining/data  gpu=none  numpy=2.3.5  pandas=2.3.3


In [2]:
# ── 01 · Load artifacts and verify the LargeST contract ─────────────────────
flow_df = pd.read_hdf(DATA / 'sd_his_2019.h5', key='t')
meta    = pd.read_csv(DATA / 'sd_meta.csv')
adj     = np.load(DATA / 'sd_rn_adj.npy')
idx     = {k: np.load(DATA / '2019' / f'idx_{k}.npy') for k in ('train', 'val', 'test')}
with np.load(DATA / '2019' / 'his.npz') as z:
    MEAN, STD = float(z['mean']), float(z['std'])
    his_flow = z['data'][..., 0]

F = flow_df.to_numpy(dtype=np.float32)          # raw flow, veh/5min
T, N = F.shape
TIMES = flow_df.index
LANE_COL = 'Lanes' if 'Lanes' in meta.columns else 'Lane'
LANES = meta[LANE_COL].to_numpy()
TRAIN_END = int(idx['val'][0] - T_IN)           # LargeST fits its scaler on F[:TRAIN_END]

assert (T, N) == (35040, 716), (T, N)
assert list(flow_df.columns.astype(str)) == list(meta['ID'].astype(str)), 'h5 column order != sd_meta row order'
assert TIMES.tz is None and TIMES[0] == pd.Timestamp('2019-01-01 00:00')
assert (np.diff(TIMES.values) == np.timedelta64(15, 'm')).all(), 'index is not a regular 15-min grid'
assert np.abs(his_flow * STD + MEAN - F).max() < 1e-2, 'his.npz channel 0 is not the normalised h5 flow'
assert np.isclose(F[:TRAIN_END].mean(dtype=np.float64), MEAN, atol=1e-3), 'scaler mean is not train-only'
assert np.isclose(F[:TRAIN_END].std(dtype=np.float64), STD, atol=1e-3), 'scaler std is not train-only'
assert sum(len(v) for v in idx.values()) == T - T_IN - HORIZON + 1
assert all((np.diff(v) == 1).all() for v in idx.values()) and idx['val'][0] == idx['train'][-1] + 1
del his_flow

print(f'F {F.shape}  mean(train)={MEAN:.3f}  std(train)={STD:.3f}  TRAIN_END={TRAIN_END} ({TIMES[TRAIN_END]})')
for k, v in idx.items():
    print(f'{k:5s} {len(v):5d} samples   targets {TIMES[v[0] + 1]} → {TIMES[v[-1] + HORIZON]}')
print('\nmeta columns:', list(meta.columns))
print('Type values:', meta['Type'].unique().tolist() if 'Type' in meta else 'n/a')
print('lanes 1-2 / 3-4 / >=5:', (LANES <= 2).sum(), ((LANES >= 3) & (LANES <= 4)).sum(), (LANES >= 5).sum(),
      '  (paper: 73 / 449 / 194)')
print('freeways:', meta['Fwy'].value_counts().to_dict())
print('directions:', meta['Direction'].value_counts().to_dict())
off = adj.copy(); np.fill_diagonal(off, 0)
print(f'adjacency: {np.count_nonzero(off)} off-diagonal edges, mean degree {np.count_nonzero(off, 1).mean():.1f}, '
      f'symmetric={np.allclose(adj, adj.T)}, diag={np.unique(np.diag(adj))[:3]}')

F (35040, 716)  mean(train)=247.247  std(train)=184.309  TRAIN_END=21009 (2019-08-07 20:15:00)
train 21010 samples   targets 2019-01-01 03:00:00 → 2019-08-08 02:00:00
val    7003 samples   targets 2019-08-07 23:30:00 → 2019-10-20 00:45:00
test   7004 samples   targets 2019-10-19 22:15:00 → 2019-12-31 23:45:00

meta columns: ['ID', 'Lat', 'Lng', 'District', 'County', 'Fwy', 'Lanes', 'Type', 'Direction', 'ID2']
Type values: ['Mainline']
lanes 1-2 / 3-4 / >=5: 73 449 194   (paper: 73 / 449 / 194)
freeways: {'I5-N': 95, 'I5-S': 86, 'I15-N': 69, 'I15-S': 58, 'I805-S': 51, 'I805-N': 49, 'I8-W': 47, 'I8-E': 39, 'SR78-W': 24, 'SR94-W': 20, 'SR78-E': 20, 'SR125-N': 19, 'SR52-W': 18, 'SR94-E': 16, 'SR163-S': 15, 'SR56-E': 13, 'SR52-E': 13, 'SR125-S': 11, 'I905-E': 11, 'SR56-W': 11, 'I905-W': 11, 'SR163-N': 10, 'SR54-W': 3, 'SR54-E': 3, 'SR11-W': 3, 'SR11-E': 1}
directions: {'N': 242, 'S': 221, 'W': 137, 'E': 116}
adjacency: 17319 off-diagonal edges, mean degree 24.2, symmetric=False, diag=[1.]


In [3]:
# ── 02 · Audit: zeros (missing-data sentinel vs genuine zero flow) ──────────
def runs(mask):
    """Runs of True along time for a (T, N) bool array -> (sensor, start, length) arrays."""
    d = np.diff(np.pad(mask.T.astype(np.int8), ((0, 0), (1, 1))), axis=1)
    s_sensor, s_t = np.nonzero(d == 1)
    _, e_t = np.nonzero(d == -1)
    return s_sensor, s_t, e_t - s_t

Z = F == 0
zero_frac = Z.mean(0)
DEAD = np.nonzero(zero_frac > 0.95)[0]
LIVE = zero_frac <= 0.95
SYS_OUT = Z[:, LIVE].mean(1) > 0.9                     # network-wide outage timesteps
Zc = Z & LIVE[None, :] & ~SYS_OUT[:, None]              # zeros on live sensors, outside outages

rs, rt, rl = runs(Z)
cells_by_len = {f'{a}-{b}': int(rl[(rl >= a) & (rl <= b)].sum()) for a, b in
                [(1, 1), (2, 4), (5, 96), (97, 10**6)]}

A_nb = (adj > 0) & ~np.eye(N, dtype=bool)              # graph neighbours, self excluded
nz = (~Z).astype(np.float32)
nb_mean = (F @ A_nb.T.astype(np.float32)) / np.maximum(nz @ A_nb.T.astype(np.float32), 1)
nb_ok = nb_mean[Zc] > 50                               # neighbours carrying normal traffic
prev_next_high = np.zeros_like(Z)
prev_next_high[1:-1] = (F[:-2] > 100) & (F[2:] > 100)

hour = TIMES.hour.to_numpy()
zero_by_hour = pd.Series(Zc.sum(1)).groupby(hour).sum() / (LIVE.sum() * (T / 24))

audit_zero = {
    'overall_zero_frac': float(Z.mean()),
    'zero_frac_live_excl_outages': float(Zc.sum() / (LIVE.sum() * T)),
    'n_sensors_gt30': int((zero_frac > .3).sum()), 'n_sensors_gt50': int((zero_frac > .5).sum()),
    'dead_sensors': [int(s) for s in DEAD], 'dead_ids': meta['ID'].iloc[DEAD].tolist(),
    'n_outage_steps': int(SYS_OUT.sum()), 'outage_times': [str(t) for t in TIMES[SYS_OUT][:40]],
    'zero_cells_by_run_length': cells_by_len,
    'frac_zero_cells_with_neighbours_flowing(>50)': float(nb_ok.mean()),
    'frac_zero_cells_isolated_between_high_flow(>100)': float(prev_next_high[Zc].mean()),
}
print(json.dumps(audit_zero, indent=1, default=str))
print('\nzero rate by hour (live sensors, outages excluded):')
print(zero_by_hour.round(4).to_string())

{
 "overall_zero_frac": 0.02247395634039948,
 "zero_frac_live_excl_outages": 0.01962930720233299,
 "n_sensors_gt30": 32,
 "n_sensors_gt50": 10,
 "dead_sensors": [
  146,
  147
 ],
 "dead_ids": [
  1119528,
  1119842
 ],
 "n_outage_steps": 4,
 "outage_times": [
  "2019-03-10 02:00:00",
  "2019-03-10 02:15:00",
  "2019-03-10 02:30:00",
  "2019-03-10 02:45:00"
 ],
 "zero_cells_by_run_length": {
  "1-1": 5072,
  "2-4": 10127,
  "5-96": 17851,
  "97-1000000": 530791
 },
 "frac_zero_cells_with_neighbours_flowing(>50)": 0.8666312357843766,
 "frac_zero_cells_isolated_between_high_flow(>100)": 0.0011056878783621158
}

zero rate by hour (live sensors, outages excluded):
0     0.0216
1     0.0230
2     0.0238
3     0.0233
4     0.0204
5     0.0189
6     0.0189
7     0.0194
8     0.0189
9     0.0188
10    0.0188
11    0.0188
12    0.0188
13    0.0188
14    0.0187
15    0.0187
16    0.0187
17    0.0187
18    0.0187
19    0.0187
20    0.0187
21    0.0188
22    0.0192
23    0.0200


In [4]:
# ── 03 · Audit: the 999 ceiling, physical plausibility, stuck sensors ───────
C999 = F == 999
top = np.bincount(np.clip(F, 0, 1000).astype(np.int64).ravel(), minlength=1001)
print(f'cells == 999: {C999.sum()}   cells > 999: {(F > 999).sum()}')
print('value counts 990..999:', dict(zip(range(990, 1000), top[990:1000].tolist())))

rs9, rt9, rl9 = runs(C999)
rows = []
for s in np.nonzero(C999.any(0))[0]:
    cells = C999[:, s]
    rows.append({'sensor': int(s), 'ID': int(meta['ID'].iat[s]), 'Fwy': meta['Fwy'].iat[s], 'Dir': meta['Direction'].iat[s],
                 'lanes': int(LANES[s]), 'n999': int(cells.sum()),
                 'implied_vphpl': round(999 * 12 / LANES[s]),
                 'max_run': int(rl9[rs9 == s].max()),
                 'peak_hour_share': float(np.isin(hour[cells], [6, 7, 8, 15, 16, 17, 18]).mean()),
                 'nb_flow_median': float(np.median(nb_mean[cells, s])),
                 'verdict': 'corrupt' if LANES[s] <= 4 else 'censored'})
t999 = pd.DataFrame(rows)
if len(t999):
    print(f'{len(t999)} sensors report 999 (top 40):')
    print(t999.sort_values('n999', ascending=False).head(40).to_string(index=False))
    print('999s by hour:', pd.Series(hour[np.nonzero(C999)[0]]).value_counts().sort_index().to_dict())

INVALID = C999 & (LANES <= 4)[None, :]        # >= 3,000 veh/h/lane: physically impossible
CENSORED = C999 & (LANES >= 5)[None, :]       # plausible saturation; value is a lower bound
vphpl = F * 12 / LANES[None, :]
print('\ncells above implied veh/h/lane:', {thr: int((vphpl > thr).sum()) for thr in (2400, 2800, 3200)},
      '— for decision at Gate 0; only the 999 rule is applied now')
del vphpl

same = np.zeros_like(Z)
same[1:] = (F[1:] == F[:-1]) & (F[1:] > 0) & (F[1:] != 999)
ss, st, sl = runs(same)
STUCK = np.zeros_like(Z)
for s, t0, L in zip(ss[sl >= 3], st[sl >= 3], sl[sl >= 3]):   # >= 4 identical values = 1 h
    STUCK[t0 - 1:t0 + L, s] = True
print(f'stuck runs (identical non-zero, non-999 values): >=1h {int((sl >= 3).sum())}, >=2h {int((sl >= 7).sum())}; '
      f'cells {int(STUCK.sum())}; sensors {int(STUCK.any(0).sum())}')

audit_999 = {'n999': int(C999.sum()), 'n_gt999': int((F > 999).sum()), 'n_invalid': int(INVALID.sum()),
             'n_censored': int(CENSORED.sum()), 'per_sensor': rows,
             'n_stuck_runs_1h': int((sl >= 3).sum()), 'n_stuck_cells': int(STUCK.sum())}

cells == 999: 37   cells > 999: 0
value counts 990..999: {990: 51, 991: 55, 992: 54, 993: 46, 994: 76, 995: 37, 996: 35, 997: 28, 998: 39, 999: 37}
7 sensors report 999 (top 40):
 sensor      ID   Fwy Dir  lanes  n999  implied_vphpl  max_run  peak_hour_share  nb_flow_median  verdict
    366 1122479 I15-S   S      7    16           1713        1         1.000000      780.318176 censored
    293 1120611 I15-N   N      8    10           1498        1         1.000000      621.632141 censored
    350 1117931 I15-S   S      6     6           1998        1         0.333333      259.843445 censored
    368 1115838 I15-S   S      6     2           1998        1         1.000000      651.964233 censored
     21 1117748  I5-N   N      6     1           1998        1         1.000000      401.136353 censored
    204 1115463  I8-E   E      6     1           1998        1         1.000000      393.000000 censored
    307 1123003 I15-N   N      6     1           1998        1         1.000000      4


cells above implied veh/h/lane: {2400: 11954, 2800: 9260, 3200: 5049} — for decision at Gate 0; only the 999 rule is applied now
stuck runs (identical non-zero, non-999 values): >=1h 5506, >=2h 458; cells 31011; sensors 507


In [5]:
# ── 04 · Audit: DST convention of the (naive) timestamp index ───────────────
# Commuters follow the wall clock, so on a wall-clock index the AM ramp does not move
# across a DST switch; on a fixed-offset index it jumps by 60 min.
Fn = np.where(Z, np.nan, F)[:, LIVE]
net = np.nanmean(Fn, axis=1)
del Fn
slot = (TIMES.hour * 4 + TIMES.minute // 15).to_numpy()
dates = TIMES.normalize()

def weekday_profile(days):
    sel = dates.isin(days)
    return pd.Series(net[sel]).groupby(slot[sel]).mean().to_numpy()

def am_ramp_time(p):
    """Minutes after midnight where the 03:00-10:00 profile first crosses 50% of its rise."""
    w = p[12:40]
    lo, hi = w.min(), w.max()
    k = max(int(np.argmax(w >= lo + 0.5 * (hi - lo))), 1)
    frac = (lo + 0.5 * (hi - lo) - w[k - 1]) / (w[k] - w[k - 1])
    return (12 + k - 1 + frac) * 15

dst = {}
for name, day in [('spring', '2019-03-10'), ('fall', '2019-11-03')]:
    d0 = pd.Timestamp(day)
    wk = pd.bdate_range(d0 - pd.Timedelta(days=16), d0 + pd.Timedelta(days=16))
    before, after = wk[wk < d0][-10:], wk[wk > d0][:10]
    dst[name] = am_ramp_time(weekday_profile(after)) - am_ramp_time(weekday_profile(before))

gap = (TIMES >= '2019-03-10 02:00') & (TIMES < '2019-03-10 03:00')
ctrl = (TIMES.hour == 2) & dates.isin(pd.to_datetime(['2019-03-03', '2019-03-17']))
dup = (TIMES >= '2019-11-03 01:00') & (TIMES < '2019-11-03 02:00')
dup_ctrl = (TIMES.hour == 1) & dates.isin(pd.to_datetime(['2019-10-27', '2019-11-10']))

if abs(dst['spring']) < 25 and abs(dst['fall']) < 25:
    TRAFFIC_CLOCK = 'wallclock'
elif dst['spring'] < -35 and dst['fall'] > 35:
    TRAFFIC_CLOCK = 'fixed_offset'
else:
    TRAFFIC_CLOCK = 'inconclusive'

audit_dst = {'am_ramp_shift_min': {k: round(float(v), 1) for k, v in dst.items()},
             'zero_frac_2019-03-10_02h': float(Z[gap][:, LIVE].mean()),
             'zero_frac_control_02h': float(Z[ctrl][:, LIVE].mean()),
             'flow_ratio_2019-11-03_01h_vs_control': float(np.nanmean(net[dup]) / np.nanmean(net[dup_ctrl])),
             'verdict': TRAFFIC_CLOCK}
print(json.dumps(audit_dst, indent=1))

MISSING = Z                     # decision per proposal: every zero on a mainline sensor is missing
save_factor('audit_masks', missing=MISSING, invalid999=INVALID, censored999=CENSORED, stuck=STUCK,
            dead=DEAD, outage=SYS_OUT)
save_result('audit', {'zero': audit_zero, '999': audit_999, 'dst': audit_dst})
print('saved factors/audit_masks.npz and results/audit.json')

{
 "am_ramp_shift_min": {
  "spring": 0.3,
  "fall": -2.5
 },
 "zero_frac_2019-03-10_02h": 1.0,
 "zero_frac_control_02h": 0.015581232492997199,
 "flow_ratio_2019-11-03_01h_vs_control": 0.8921847939491272,
 "verdict": "wallclock"
}


saved factors/audit_masks.npz and results/audit.json


In [6]:
# ── 05 · Evaluator (LargeST protocol) + HL reproduction + historical-average baseline
# Metrics follow src/base/engine.py + src/utils/metrics.py exactly: cells whose raw
# label is 0 are excluded; per-horizon MAE/RMSE/MAPE, 'avg' = mean over the 12 horizons.
def horizon_metrics(pred_fn, sample_idx, extra_mask=None):
    """pred_fn(sample_idx, h) -> (S, N) raw-flow prediction for target time t+h."""
    rows = []
    for h in range(1, HORIZON + 1):
        y = F[sample_idx + h].astype(np.float64)
        m = y != 0
        if extra_mask is not None:
            m &= extra_mask(sample_idx, h)
        err = pred_fn(sample_idx, h).astype(np.float64)[m] - y[m]
        rows.append((h, np.abs(err).mean(), np.sqrt((err ** 2).mean()), (np.abs(err) / y[m]).mean()))
    out = pd.DataFrame(rows, columns=['h', 'MAE', 'RMSE', 'MAPE']).set_index('h')
    out.loc['avg'] = out.mean()
    return out

PAPER_HL = {3: (33.61, 50.97, .2077), 6: (57.80, 84.92, .3773), 12: (101.74, 140.14, .7684),
            'avg': (60.79, 87.40, .4188)}
hl = lambda ix, h: F[ix]
test_exact = idx['test'][: len(idx['test']) // 64 * 64]      # LargeST's loader drops the last partial batch
hl_exact = horizon_metrics(hl, test_exact)
cmp = pd.DataFrame({k: [*hl_exact.loc[k], *v] for k, v in PAPER_HL.items()},
                   index=['MAE', 'RMSE', 'MAPE', 'paper_MAE', 'paper_RMSE', 'paper_MAPE']).T
print('HL, LargeST-exact sample set:\n', cmp.round(4).to_string())
HL_OK = abs(cmp.loc['avg', 'MAE'] - 60.79) < 0.05 and abs(cmp.loc[3, 'MAE'] - 33.61) < 0.05
print('HL REPRODUCTION:', 'PASS' if HL_OK else 'MISMATCH — data differs from the paper; investigate before training')

# Historical average: train-only median per (day type, 15-min slot, sensor), zeros/corrupt excluded
dtype3 = np.select([TIMES.dayofweek < 5, TIMES.dayofweek == 5], [0, 1], 2)
valid = ~(MISSING | INVALID)
Ftr = np.where(valid, F, np.nan)[:TRAIN_END]
PROFILE = np.full((3, STEPS_PER_DAY, N), np.nan, np.float32)
for d in range(3):
    for k in range(STEPS_PER_DAY):
        sel = (dtype3[:TRAIN_END] == d) & (slot[:TRAIN_END] == k)
        PROFILE[d, k] = np.nanmedian(Ftr[sel], axis=0)
del Ftr
PROFILE = np.where(np.isnan(PROFILE), np.nanmedian(PROFILE, axis=(0, 1), keepdims=True), PROFILE)
PROFILE = np.nan_to_num(PROFILE, nan=0.0)                     # fully dead sensors
save_factor('profile_median_train', profile=PROFILE)
ha = lambda ix, h: PROFILE[dtype3[ix + h], slot[ix + h]]

res = {'HL': horizon_metrics(hl, idx['test']), 'HA': horizon_metrics(ha, idx['test']),
       'HA_val': horizon_metrics(ha, idx['val'])}
for k, v in res.items():
    print(f'\n{k} (full sample set):\n', v.loc[[1, 2, 3, 4, 6, 8, 12, 'avg']].round(3).T.to_string())

HL, LargeST-exact sample set:
           MAE      RMSE    MAPE  paper_MAE  paper_RMSE  paper_MAPE
3     33.6113   50.9749  0.2077      33.61       50.97      0.2077
6     57.7979   84.9242  0.3773      57.80       84.92      0.3773
12   101.7380  140.1406  0.7684     101.74      140.14      0.7684
avg   60.7892   87.3996  0.4188      60.79       87.40      0.4188
HL REPRODUCTION: PASS



HL (full sample set):
 h          1       2       3       4       6        8       12     avg
MAE   17.722  25.734  33.611  41.389  57.799   72.897  101.778  60.800
RMSE  27.311  39.584  50.957  62.015  84.877  104.683  140.100  87.363
MAPE   0.107   0.156   0.207   0.261   0.377    0.498    0.768   0.418

HA (full sample set):
 h          1       2       3       4       6       8      12     avg
MAE   29.774  29.780  29.784  29.787  29.790  29.792  29.799  29.789
RMSE  57.035  57.039  57.043  57.045  57.046  57.048  57.049  57.045
MAPE   0.194   0.194   0.194   0.194   0.195   0.195   0.195   0.195

HA_val (full sample set):
 h          1       2       3       4       6       8      12     avg
MAE   21.861  21.861  21.860  21.860  21.861  21.863  21.863  21.862
RMSE  41.030  41.030  41.029  41.029  41.030  41.032  41.035  41.031
MAPE   0.128   0.128   0.128   0.128   0.128   0.128   0.128   0.128


In [7]:
# ── 06 · Gate 0 summary ─────────────────────────────────────────────────────
LSTM_BASELINE = {1: 13.72, 2: 16.65, 4: 21.16, 6: 25.79, 8: 29.68, 12: 37.45, 'avg': 26.35}
GWNET_PAPER   = {3: 15.24, 6: 17.74, 12: 21.56, 'avg': 17.74}
cols = [1, 2, 3, 4, 6, 8, 12, 'avg']
rows = {'HL (ours)': res['HL']['MAE'], 'HA profile (ours)': res['HA']['MAE'],
        'LSTM (yours)': LSTM_BASELINE, 'GWNet (paper)': GWNET_PAPER}
summary = pd.DataFrame({k: pd.Series(v).reindex(cols) for k, v in rows.items()}, index=cols).T
print('Test MAE by horizon (LargeST protocol)\n', summary.round(2).to_string())

beats = [h for h in LSTM_BASELINE if h != 'avg' and res['HA']['MAE'].loc[h] < LSTM_BASELINE[h]]
print(f'\nHA profile beats the LSTM at horizons: {beats or "none"}')
print(f'dead sensors: {len(DEAD)}   corrupt 999 cells: {int(INVALID.sum())}   censored 999 cells: {int(CENSORED.sum())}'
      f'   stuck cells: {int(STUCK.sum())}   DST verdict: {TRAFFIC_CLOCK}   HL reproduction: {"PASS" if HL_OK else "MISMATCH"}')
save_result('gate0', {'mae_table': summary.to_dict(), 'hl_exact': hl_exact.to_dict(),
                      'ha_beats_lstm_at': beats, 'dst': TRAFFIC_CLOCK, 'hl_reproduced': bool(HL_OK)})

Test MAE by horizon (LargeST protocol)
                        1      2      3      4      6      8      12    avg
HL (ours)          17.72  25.73  33.61  41.39  57.80  72.90  101.78  60.80
HA profile (ours)  29.77  29.78  29.78  29.79  29.79  29.79   29.80  29.79
LSTM (yours)       13.72  16.65    NaN  21.16  25.79  29.68   37.45  26.35
GWNet (paper)        NaN    NaN  15.24    NaN  17.74    NaN   21.56  17.74

HA profile beats the LSTM at horizons: [12]
dead sensors: 2   corrupt 999 cells: 0   censored 999 cells: 37   stuck cells: 31011   DST verdict: wallclock   HL reproduction: PASS


In [8]:
# ── 06b · Gate 0 follow-up: HA val→test gap, lane-count plausibility, stuck runs (CPU, <1 min)
# (a) HA error is horizon-independent, so h=1 localises it by day and by sensor.
def ha_err(split):
    ix = idx[split]
    y = F[ix + 1]
    e = np.where(y != 0, np.abs(PROFILE[dtype3[ix + 1], slot[ix + 1]] - y), np.nan)
    day = TIMES[ix + 1].normalize()
    ok = ~np.isnan(e)
    by_day = pd.Series(np.nansum(e, 1)).groupby(day).sum() / pd.Series(ok.sum(1)).groupby(day).sum()
    by_sensor = np.nansum(e, 0) / np.maximum(ok.sum(0), 1)
    return e, by_day, by_sensor

e_val, day_val, sen_val = ha_err('val')
e_test, day_test, sen_test = ha_err('test')
worst = day_test.sort_values(ascending=False).head(12)
print('HA test MAE by day — worst 12 (median day %.1f):' % day_test.median())
print('  ' + '  '.join(f'{d:%a %b %d}={v:.1f}' for d, v in worst.items()))
hol_weeks = (day_test.index >= '2019-11-25') & (day_test.index <= '2019-11-29') | (day_test.index >= '2019-12-21')
print(f'HA test MAE: all days {np.nanmean(e_test):.2f} | excl. Thanksgiving week & Dec 21-31 '
      f'{np.nanmean(e_test[~TIMES[idx["test"] + 1].normalize().isin(day_test.index[hol_weeks])]):.2f} | val {np.nanmean(e_val):.2f}')

nzF = np.where(MISSING, np.nan, F)
lvl_train = np.nanmean(nzF[:TRAIN_END], 0)
lvl_test = np.nanmean(nzF[idx['test'][0]:], 0)
ratio = lvl_test / lvl_train
sens = pd.DataFrame({'ID': meta['ID'], 'Fwy': meta['Fwy'], 'lanes': LANES, 'HA_val': sen_val, 'HA_test': sen_test,
                  'test/train level': ratio, 'zero_frac_test': MISSING[idx['test'][0]:].mean(0)})
sens['gap'] = sens['HA_test'] - sens['HA_val']
print(f'\nsensors with test/train level shift >20%: {int((np.abs(np.log(ratio)) > np.log(1.2)).sum())}   '
      f'HA test MAE without the 20 worst-gap sensors: '
      f'{np.nanmean(np.delete(e_test, sens["gap"].nlargest(20).index, axis=1)):.2f}')
print(sens.sort_values('gap', ascending=False).head(15).round(2).to_string())

# (b) Lane counts vs the flows each sensor actually carries (train p99, zeros excluded)
p99 = np.nanpercentile(nzF[:TRAIN_END], 99, axis=0)
lane_tab = pd.DataFrame({'ID': meta['ID'], 'Fwy': meta['Fwy'], 'lanes': LANES, 'p99_flow': p99,
                         'p99_vphpl': p99 * 12 / LANES, 'lanes_needed@2200': np.ceil(p99 * 12 / 2200)})
suspect = lane_tab[lane_tab['p99_vphpl'] > 2600].sort_values('p99_vphpl', ascending=False)
print(f'\nsensors whose train p99 implies >2,600 veh/h/lane: {len(suspect)}')
print(suspect.head(20).round(0).to_string())
del nzF

# (c) Stuck runs of >= 2 h (8 identical rounded values): level, time of day, proximity to outages
long = sl >= 7
vals, hrs = F[st[long], ss[long]], hour[st[long]]
near_zero = np.array([MISSING[max(t - 2, 0):t + L + 2, s].any() for s, t, L in zip(ss[long], st[long], sl[long])])
print(f'\n>=2h stuck runs: {int(long.sum())} on {len(np.unique(ss[long]))} sensors; value quantiles '
      f'{np.percentile(vals, [5, 50, 95]).round(0).tolist()}; start-hour counts {pd.Series(hrs).value_counts().sort_index().to_dict()}; '
      f'adjacent to an outage: {near_zero.mean():.2f}')

save_result('gate0b', {'ha_worst_days': {str(k): v for k, v in worst.items()},
                       'n_level_shift_20pct': int((np.abs(np.log(ratio)) > np.log(1.2)).sum()),
                       'lane_suspect_ids': suspect['ID'].tolist(), 'n_stuck_2h': int(long.sum())})

HA test MAE by day — worst 12 (median day 23.0):
  Wed Dec 25=127.2  Thu Nov 28=121.9  Fri Nov 29=77.6  Tue Dec 24=68.8  Thu Dec 26=61.0  Tue Dec 31=54.7  Wed Nov 27=43.1  Mon Dec 23=42.0  Fri Dec 27=41.6  Wed Dec 04=40.0  Mon Nov 11=39.6  Mon Dec 30=38.0
HA test MAE: all days 29.77 | excl. Thanksgiving week & Dec 21-31 23.85 | val 21.86



sensors with test/train level shift >20%: 19   HA test MAE without the 20 worst-gap sensors: 28.31
          ID      Fwy  lanes  HA_val  HA_test  test/train level  zero_frac_test     gap
147  1119842     I5-S      5    0.00   365.11               NaN            0.99  365.11
146  1119528     I5-S      5    0.00   336.82               NaN            0.99  336.82
295  1117909    I15-N      6   19.65   131.07              0.59            0.00  111.42
240  1108333     I8-W      2  138.12   224.81              0.36            0.01   86.70
62   1122507     I5-N      4   57.99   141.03              0.59            0.00   83.04
294  1115721    I15-N      7   21.91    77.30              0.85            0.00   55.39
154  1108477     I5-S      4   45.84    99.64              0.70            0.00   53.80
70   1108663     I5-N      4   47.25    92.17              0.75            0.00   44.93
61   1119659     I5-N      4   28.66    73.13              0.83            0.00   44.47
342  1117920    I15-


>=2h stuck runs: 458 on 28 sensors; value quantiles [1.0, 57.0, 57.0]; start-hour counts {0: 152, 1: 100, 2: 39, 3: 16, 4: 8, 5: 2, 6: 1, 7: 2, 8: 5, 11: 6, 13: 1, 18: 2, 20: 4, 21: 12, 22: 44, 23: 64}; adjacent to an outage: 0.05


## ⛔ GATE 0 — STOP

Gate 0 passed: HL matches the paper to 4 decimals, and the DST index is local wall-clock.

Proposed decisions, awaiting go-ahead:
1. **999s**: keep as-is. There are 37 cells, all on 6–8-lane sensors, with no pile-up at the cap. Report it as tail censoring.
2. **Lane counts**: only sensor 1108333 (I8-W, listed with 2 lanes, carries 5 lanes' worth of flow) is implausible. Use an effective lane count of 5 for per-lane features.
3. **Stuck runs of ≥2 h** (458 runs on 28 sensors, mostly overnight): treat as missing. Add them to the input mask and exclude them from the clean protocol.
4. **Sensors whose level shifted in test** (19 with >20% change) and the 2 dead sensors: keep them in the LargeST protocol. Also report clean and stable-sensor metrics. Use the test-derived flag for reporting only, never for training.
5. **Missing-input imputation (R2)**: fill with the trailing 7-day same-slot median, which only looks backwards and adapts to drift. Fall back to the train profile.
6. **Evaluation**: always stratify out holiday weeks. The rain-vs-dry comparison excludes holiday days, because the Thanksgiving storm overlaps the holiday.

Stage 1 (GPU overfit test) cells are added after go-ahead.

## Stage 1 · Overfit test (Colab T4 ~2 min · M2 MPS ~10 min)

On Colab: **Runtime ▸ Change runtime type ▸ T4 GPU**. On the M2 it uses the Apple GPU (MPS) automatically. In a fresh runtime, run **00** and **01** first. Cells 02–06b are not needed, because their results are already on Drive.

- **07** writes `conglib.py`, the shared model and training code. The notebook imports it now, and the backgrounded `train.py` will import it from Stage 3.
- **08** loads LargeST's three input channels onto the GPU and checks the window and channel alignment.
- **09** trains GWNet on 100 training windows for 50 epochs, with dropout and weight decay off. The loss must collapse.

In [3]:
%%writefile {WORK}/cong/conglib.py
# ── 07 · conglib.py — shared model/training code (notebook + train.py) ─────
"""Shared training code for congestion_sd — imported by the notebook and by train.py (nohup runs).

Runs on CUDA (Colab T4), Apple MPS (M2) or CPU."""
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch

T_IN, HORIZON = 12, 12


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device():
    if os.environ.get('CONG_DEVICE'):                      # e.g. CONG_DEVICE=cpu for bitwise-reproducible tests
        return torch.device(os.environ['CONG_DEVICE'])
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()


def reset_mem(device):
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    elif device.type == 'mps':
        torch.mps.empty_cache()


def mem_gib(device):
    """CUDA: peak allocated since reset_mem. MPS: memory the Metal driver currently holds (incl. cache). CPU: nan."""
    if device.type == 'cuda':
        return torch.cuda.max_memory_allocated() / 2**30
    if device.type == 'mps':
        return torch.mps.driver_allocated_memory() / 2**30
    return float('nan')


def is_oom(err):
    return isinstance(err, torch.cuda.OutOfMemoryError) or 'out of memory' in str(err).lower()


class Windows:
    """Whole (T, N, C) input tensor and raw (T, N) flow targets on one device; batches are index gathers.

    Window semantics match LargeST's DataLoader: x = inputs[t-11 .. t], y = flow[t+1 .. t+12].
    Targets live in their own array, so no covariate channel can ever reach the loss or the scaler.
    """

    def __init__(self, X, Y, device, x_dtype=torch.float32, Yc=None, time=None, static=None, wx=None, wx_w=None):
        # Optional factored inputs, joined per batch so the full (T, N, C) tensor never exists:
        # time (T, Ct) shared by all sensors, static (N, Cs) shared by all steps, wx (T, S, k) weather stations
        # mixed onto sensors by wx_w (N, S).
        self.X = torch.as_tensor(np.asarray(X), dtype=x_dtype).to(device)
        self.Y = torch.as_tensor(np.asarray(Y), dtype=torch.float32).to(device)
        self.Yc = None if Yc is None else torch.as_tensor(np.asarray(Yc), dtype=torch.long).to(device)
        f32 = lambda a: None if a is None else torch.as_tensor(np.asarray(a), dtype=torch.float32).to(device)
        self.Xt, self.Xs, self.Ww = f32(time), f32(static), f32(wx_w)
        self.Xw = None if wx is None else torch.as_tensor(np.asarray(wx), dtype=torch.float16).to(device)
        self.x_off = torch.arange(-(T_IN - 1), 1, device=device)
        self.y_off = torch.arange(1, HORIZON + 1, device=device)
        self.n_channels = (self.X.shape[-1] + (0 if self.Xt is None else self.Xt.shape[-1]) +
                           (0 if self.Xs is None else self.Xs.shape[-1]) + (0 if self.Xw is None else self.Xw.shape[-1]))

    def batch(self, starts):
        starts = torch.as_tensor(np.asarray(starts), device=self.X.device)
        ti = starts[:, None] + self.x_off
        parts = [self.X[ti].float()]                          # (B, T_IN, N, Cd)
        b, L, n = ti.shape[0], ti.shape[1], self.X.shape[1]
        if self.Xt is not None:
            parts.append(self.Xt[ti][:, :, None, :].expand(b, L, n, -1))
        if self.Xs is not None:
            parts.append(self.Xs[None, None].expand(b, L, -1, -1))
        if self.Xw is not None:
            parts.append(torch.einsum('blsk,ns->blnk', self.Xw[ti].float(), self.Ww))
        x = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)      # (B, T_IN, N, C)
        y = self.Y[starts[:, None] + self.y_off]             # (B, HORIZON, N) raw veh/5min
        if self.Yc is None:
            return x, y
        return x, y, self.Yc[starts[:, None] + self.y_off]   # + (B, HORIZON, N) class labels, -1 = ignore


def masked_mae(pred, y):
    """LargeST's masked MAE: cells whose raw label is 0 (missing) are excluded."""
    m = (y != 0).float()
    return ((pred - y).abs() * m).sum() / m.sum().clamp_min(1)


def build_gwnet(adj, adj_type, adp_adj, input_dim, device, repo, dropout=0.3,
                init_dim=32, skip_dim=256, end_dim=512):
    """LargeST's GWNET with its default SD hyper-parameters; adj_type='none' drops the road graph."""
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from src.models.gwnet import GWNET
    from src.utils.graph_algo import normalize_adj_mx
    mats = [] if adj_type == 'none' else normalize_adj_mx(adj, adj_type)
    supports = [torch.tensor(np.asarray(a), dtype=torch.float32, device=device) for a in mats]
    return GWNET(node_num=adj.shape[0], input_dim=input_dim, output_dim=1, supports=supports,
                 adp_adj=adp_adj, dropout=dropout, residual_channels=init_dim, dilation_channels=init_dim,
                 skip_channels=skip_dim, end_channels=end_dim).to(device)


def forward_raw(model, x, mean, std, amp):
    """Model output (B, HORIZON, N, 1) -> raw-scale flow (B, HORIZON, N) in fp32."""
    with torch.autocast(x.device.type, dtype=torch.float16, enabled=amp):
        out = model(x)
    return out.float().squeeze(-1) * std + mean


def train_step(model, opt, scaler, x, y, mean, std, amp, clip=5.0, micro_bs=None):
    """One optimiser step on the batch (x, y); returns its masked MAE.

    With micro_bs, the batch is processed in chunks whose losses are each divided by the valid-cell count
    of the *whole* batch, so the accumulated gradient equals the full-batch masked-MAE gradient exactly.
    Only BatchNorm differs: it normalises over each micro-batch instead of the full batch.
    """
    model.train()
    opt.zero_grad(set_to_none=True)
    n_valid = (y != 0).sum().clamp_min(1)
    chunk = micro_bs or len(x)
    total = 0.0
    for i in range(0, len(x), chunk):
        yb = y[i:i + chunk]
        pred = forward_raw(model, x[i:i + chunk], mean, std, amp)
        loss = ((pred - yb).abs() * (yb != 0)).sum() / n_valid
        scaler.scale(loss).backward()
        total += loss.item()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
    scaler.step(opt)
    scaler.update()
    return total


@torch.no_grad()
def predict(model, data, starts, mean, std, amp, bs=64):
    """Eval-mode raw predictions and labels for the given window starts: (S, HORIZON, N) each."""
    model.eval()
    preds, labels = [], []
    for i in range(0, len(starts), bs):
        x, y = data.batch(starts[i:i + bs])
        preds.append(forward_raw(model, x, mean, std, amp))
        labels.append(y)
    return torch.cat(preds), torch.cat(labels)


@torch.no_grad()
def evaluate(model, data, starts, mean, std, amp, bs=64):
    """Per-horizon masked MAE / RMSE / MAPE over the given windows (LargeST protocol) -> (3, HORIZON) array.

    Sums are accumulated batch by batch in float64 on the CPU (MPS has no float64), so full val/test
    sets never have to sit in device memory."""
    model.eval()
    s_abs = s_sq = s_pct = cnt = 0
    for i in range(0, len(starts), bs):
        x, y = data.batch(starts[i:i + bs])
        m = y != 0
        e = torch.where(m, forward_raw(model, x, mean, std, amp) - y, torch.zeros_like(y))
        s_abs = s_abs + e.abs().sum(dim=(0, 2)).cpu().double()
        s_sq = s_sq + (e ** 2).sum(dim=(0, 2)).cpu().double()
        s_pct = s_pct + (e.abs() / torch.where(m, y, torch.ones_like(y))).sum(dim=(0, 2)).cpu().double()
        cnt = cnt + m.sum(dim=(0, 2)).cpu().double()
    cnt = cnt.clamp_min(1)
    return torch.stack([s_abs / cnt, (s_sq / cnt).sqrt(), s_pct / cnt]).numpy()


# ── multi-task (flow + congestion class) ─────────────────────────────────────
class GWNetMT(torch.nn.Module):
    """GWNet trunk with two 1x1 heads on the shared end_conv_1 features: flow (the unchanged LargeST head)
    and congestion-class logits for every sensor and horizon."""

    def __init__(self, base, n_classes=3):
        super().__init__()
        self.base, self.K = base, n_classes
        self.cls = torch.nn.Conv2d(base.end_conv_2.in_channels, base.horizon * n_classes, kernel_size=(1, 1)).to(
            next(base.parameters()).device)                   # same device as the trunk
        self._feat = None
        base.end_conv_2.register_forward_pre_hook(self._grab)

    def _grab(self, module, inputs):
        self._feat = inputs[0]

    def forward(self, x):
        flow = self.base(x)                                   # (B, HORIZON, N, 1)
        logits = self.cls(self._feat)                         # (B, HORIZON*K, N, 1)
        b, _, n, _ = logits.shape
        return flow, logits.view(b, self.base.horizon, self.K, n).permute(0, 1, 3, 2)   # (B, HORIZON, N, K)


def forward_mt(model, x, mean, std, amp):
    with torch.autocast(x.device.type, dtype=torch.float16, enabled=amp):
        flow, logits = model(x)
    return flow.float().squeeze(-1) * std + mean, logits.float()


def train_step_mt(model, opt, scaler, x, y, yc, mean, std, amp, lam, class_w, clip=5.0, micro_bs=None):
    """Masked MAE + lam * class-weighted CE (ignore -1), accumulated exactly over micro-batches:
    each term is normalised by its full-batch denominator (valid flow cells / summed class weights)."""
    model.train()
    opt.zero_grad(set_to_none=True)
    n_valid = (y != 0).sum().clamp_min(1)
    lab = yc >= 0
    w_total = class_w[yc.clamp_min(0)][lab].sum().clamp_min(1e-6)
    chunk = micro_bs or len(x)
    tot_mae = tot_ce = 0.0
    for i in range(0, len(x), chunk):
        yb, cb = y[i:i + chunk], yc[i:i + chunk]
        pred, logits = forward_mt(model, x[i:i + chunk], mean, std, amp)
        mae = ((pred - yb).abs() * (yb != 0)).sum() / n_valid
        nll = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), cb.reshape(-1),
                                                weight=class_w, ignore_index=-1, reduction='sum') / w_total
        scaler.scale(mae + lam * nll).backward()
        tot_mae += mae.item()
        tot_ce += nll.item()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
    scaler.step(opt)
    scaler.update()
    return tot_mae, tot_ce


@torch.no_grad()
def evaluate_mt(model, data, starts, mean, std, amp, bs=64, n_classes=3):
    """Flow metrics (3, HORIZON) as in evaluate(), plus a per-horizon confusion matrix (HORIZON, K, K)
    with rows = true class, columns = predicted class (argmax), over labelled cells only."""
    model.eval()
    s_abs = s_sq = s_pct = cnt = 0
    conf = torch.zeros(HORIZON, n_classes, n_classes, dtype=torch.float64)
    for i in range(0, len(starts), bs):
        x, y, yc = data.batch(starts[i:i + bs])
        pred, logits = forward_mt(model, x, mean, std, amp)
        m = y != 0
        e = torch.where(m, pred - y, torch.zeros_like(y))
        s_abs = s_abs + e.abs().sum(dim=(0, 2)).cpu().double()
        s_sq = s_sq + (e ** 2).sum(dim=(0, 2)).cpu().double()
        s_pct = s_pct + (e.abs() / torch.where(m, y, torch.ones_like(y))).sum(dim=(0, 2)).cpu().double()
        cnt = cnt + m.sum(dim=(0, 2)).cpu().double()
        yhat = logits.argmax(-1)
        for h in range(HORIZON):
            t, p = yc[:, h].reshape(-1), yhat[:, h].reshape(-1)
            keep = t >= 0
            conf[h] += torch.bincount((t[keep] * n_classes + p[keep]).cpu(), minlength=n_classes ** 2).view(
                n_classes, n_classes).double()
    cnt = cnt.clamp_min(1)
    return torch.stack([s_abs / cnt, (s_sq / cnt).sqrt(), s_pct / cnt]).numpy(), conf.numpy()


def class_scores(conf):
    """conf (..., K, K) rows = truth -> per-class precision, recall, F1 and macro-F1 (last axis = class)."""
    tp = np.diagonal(conf, axis1=-2, axis2=-1)
    prec = tp / np.maximum(conf.sum(-2), 1)
    rec = tp / np.maximum(conf.sum(-1), 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return prec, rec, f1, f1.mean(-1)


# ── Stage E · evaluation primitives (load a finished run; single-pass caching for stratified eval) ──
def load_run(run, data_dir, proj_dir, repo, device, adj, F, LAB=None):
    """Rebuild a finished run's exact model + Windows object from its own ckpt/<run>/best.pt — inference
    only, the checkpoint is only read. `adj` and `F` (raw flow targets) are passed in since they're the
    same for every run; `LAB` (congestion class labels, or None) is only used if the run is multi-task.
    Returns (model, data, mean, std, args) where args is the training-time argparse dict from the checkpoint."""
    data_dir, proj_dir = Path(data_dir), Path(proj_dir)
    ck = torch.load(proj_dir / 'ckpt' / run / 'best.pt', map_location='cpu', weights_only=False)
    args = ck['state']['args']
    args.setdefault('task', 'flow')                     # R0/A1/A3 predate --task/--inputs/--lam (train.py's own
    args.setdefault('inputs', 'R0')                      # argparse defaults — those runs are genuinely flow-only,
    args.setdefault('lam', 15.0)                         # R0-input runs, so the defaults are the correct fill-in)
    mt = args['task'] == 'mt'
    with np.load(data_dir / '2019' / 'his.npz') as z:
        mean, std = float(z['mean']), float(z['std'])
    if args['inputs'] == 'R0':
        with np.load(data_dir / '2019' / 'his.npz') as z:
            X = z['data'].astype(np.float32)
        data = Windows(X, F, device, Yc=LAB if mt else None)
        in_dim = 3
    else:
        import features
        fin = features.assemble(args['inputs'], data_dir, proj_dir / 'factors')
        data = Windows(fin['dyn'], F, device, x_dtype=torch.float16, Yc=LAB if mt else None,
                       time=fin['time'], static=fin['static'], wx=fin['wx'], wx_w=fin['wx_w'])
        in_dim = data.n_channels
    set_seed(args['seed'])
    model = build_gwnet(adj, args['adj_type'], args['adp'], input_dim=in_dim, device=device, repo=repo,
                        dropout=args['dropout'])
    if mt:
        model = GWNetMT(model).to(device)
    model.load_state_dict(ck['model'])
    model.eval()
    return model, data, mean, std, args


@torch.no_grad()
def predict_mt(model, data, starts, mean, std, amp, bs=64):
    """Like predict(), but for a multi-task model: also returns the predicted class (argmax) and the
    true class labels, so a single forward pass over the test set can drive every stratified/bootstrap
    number downstream in plain numpy. Returns (pred_flow, true_flow, pred_class, true_class), each
    (S, HORIZON, N); the class arrays are int8 (values are only -1/0/1/2)."""
    model.eval()
    pf, tf, pc, tc = [], [], [], []
    for i in range(0, len(starts), bs):
        x, y, yc = data.batch(starts[i:i + bs])
        flow, logits = forward_mt(model, x, mean, std, amp)
        pf.append(flow)
        tf.append(y)
        pc.append(logits.argmax(-1).to(torch.int8))
        tc.append(yc.to(torch.int8))
    return torch.cat(pf), torch.cat(tf), torch.cat(pc), torch.cat(tc)


Overwriting /Users/lavanyapareek/Documents/Capstone_Post_Joining/work/cong/conglib.py


In [3]:
# ── 08 · Device setup (CUDA / Apple MPS / CPU), R0 inputs, window/channel guards ──
import importlib
import torch
sys.path.insert(0, str(WORK / 'cong'))
import conglib
importlib.reload(conglib)
from conglib import Windows, build_gwnet, masked_mae, train_step, predict, evaluate, sync, reset_mem, mem_gib, is_oom

DEVICE = conglib.pick_device()
AMP = DEVICE.type == 'cuda'                          # fp16 AMP on CUDA only; MPS gains ~11%, so stay fp32
MICRO_BS = 8 if DEVICE.type == 'mps' else None        # 8 GB M2: 8 windows/pass, accumulated to the full batch
EVAL_BS = 16 if DEVICE.type == 'mps' else 64
if DEVICE.type == 'cpu':
    print('WARNING: no GPU (CUDA/MPS) — on Colab: Runtime ▸ Change runtime type ▸ T4 GPU.')

with np.load(DATA / '2019' / 'his.npz') as z:
    X_R0 = z['data'].astype(np.float32)        # (T, N, 3): normalised flow, time of day, day of week / 7
data = Windows(X_R0, F, DEVICE)                 # targets = raw flow F, stored separately from the inputs
del X_R0

# Windows must match LargeST's loader, and input channel 0 must be exactly the normalised target
s = idx['train'][[0, 5000, -1]]
x, y = data.batch(s)
assert x.shape == (3, T_IN, N, 3) and y.shape == (3, HORIZON, N)
for j, t in enumerate(s):
    assert torch.equal(y[j].cpu(), torch.from_numpy(F[t + 1:t + 1 + HORIZON])), 'label window misaligned'
    assert np.allclose(x[j, :, :, 0].cpu().numpy() * STD + MEAN, F[t - T_IN + 1:t + 1], atol=1e-2), 'input window misaligned'
assert idx['test'][-1] + HORIZON < T
print(f'device={DEVICE} ({GPU})  amp={AMP}  micro_bs={MICRO_BS}  eval_bs={EVAL_BS}  torch={torch.__version__}  '
      f'inputs on device {tuple(data.X.shape)} = {data.X.element_size() * data.X.nelement() / 2**20:.0f} MiB  guards OK')

device=mps (none)  amp=False  micro_bs=8  eval_bs=16  torch=2.14.0  inputs on device (35040, 716, 3) = 287 MiB  guards OK


In [11]:
# ── 09 · Stage 1 · overfit tests (no dropout, no weight decay) ──────────────
# A (pass/fail): 4 windows, full batch — fewer targets (4x12x716) than parameters, so the loss
#   must keep falling far below its start; a model that can't memorise one batch has broken plumbing.
# B (your spec): 100 windows, 50 epochs — 859K targets > ~304K parameters, so it plateaus by design;
#   a sanity curve that must end below HL on the same windows.
# Gradient flow is checked in a separate fp32 pass: under AMP the first steps can overflow, the scaler
# skips them, and clipping an inf norm zeroes every finite gradient — normal, but it fools a grad check.
def fresh_model():
    conglib.set_seed(SEED)
    return build_gwnet(adj, 'doubletransition', adp_adj=1, input_dim=3, device=DEVICE, repo=REPO, dropout=0.0)

def run(starts, steps_or_epochs, bs, lr, by_epoch):
    model = fresh_model()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)
    rng = np.random.default_rng(SEED)
    curve, t0, n_steps = [], time.time(), 0
    for _ in range(steps_or_epochs):
        order = rng.permutation(len(starts)) if by_epoch else np.arange(len(starts))
        losses = []
        for i in range(0, len(starts), bs):
            losses.append(train_step(model, opt, scaler, *data.batch(starts[order[i:i + bs]]), MEAN, STD, AMP,
                                     micro_bs=MICRO_BS))
            n_steps += 1
        curve.append(float(np.mean(losses)))
    ms = 1000 * (time.time() - t0) / n_steps
    eval_mae = masked_mae(*predict(model, data, starts, MEAN, STD, AMP, bs=EVAL_BS)).item()
    return curve, eval_mae, ms

def grad_flow(starts):
    """One fp32 forward/backward — no AMP scaler, no clipping. Which parameters receive gradient?"""
    model = fresh_model()
    model.train()
    x, y = data.batch(starts)
    masked_mae(conglib.forward_raw(model, x, MEAN, STD, amp=False), y).backward()
    last = model.blocks * model.layers - 1          # GWNet's output uses only skip connections, so the
    dead = [n for n, _ in model.named_parameters()  # final layer's gconv/bn never reach the loss
            if n.startswith((f'gconv.{last}.', f'bn.{last}.'))]
    no_grad = [n for n, p in model.named_parameters() if p.grad is None or p.grad.abs().sum() == 0]
    return sorted(set(no_grad) - set(dead)), dead, sum(p.numel() for p in model.parameters())

reset_mem(DEVICE)
pool = np.random.default_rng(SEED).choice(idx['train'], 100, replace=False)
sA, sB = np.sort(pool[:4]), np.sort(pool)

unexpected, dead, n_params = grad_flow(sA)
curveA, evalA, msA = run(sA, 1000, 4, 3e-3, by_epoch=False)
curveB, evalB, msB = run(sB, 50, 20, 1e-3, by_epoch=True)
_, yB = data.batch(sB)
hlB = masked_mae(torch.as_tensor(F[sB], device=DEVICE)[:, None, :].expand_as(yB), yB).item()

print(f'params={n_params:,}   dead by design (final layer gconv/bn never reach the output): {dead}')
print(f'A · 4 windows, full batch, 1000 steps ({msA:.0f} ms/step): ' +
      '  '.join(f'{k}:{curveA[k - 1]:.1f}' for k in (1, 50, 100, 250, 500, 750, 1000)) +
      f'  | eval-mode {evalA:.2f}')
print(f'B · 100 windows, 50 epochs, bs 20 ({msB:.0f} ms/step): ' +
      '  '.join(f'{e}:{curveB[e - 1]:.1f}' for e in (1, 2, 5, 10, 20, 30, 40, 50)) +
      f'  | eval-mode {evalB:.2f} | HL same windows {hlB:.2f}')
print(f'device memory ({DEVICE.type}): {mem_gib(DEVICE):.2f} GiB')

checks = {
    'A: final < 10% of step 1': curveA[-1] < 0.1 * curveA[0],
    'A: still improving in the 2nd half (last 100 steps < 80% of steps 401-500)': np.mean(curveA[-100:]) < 0.8 * np.mean(curveA[400:500]),
    'A: eval-mode within 25% of train': abs(evalA - curveA[-1]) < 0.25 * curveA[-1],
    'B: final < HL on same windows': curveB[-1] < hlB,
    'B: final < 40% of epoch 1': curveB[-1] < 0.4 * curveB[0],
    'every parameter except dead-by-design gets gradient (fp32 pass)': not unexpected,
}
for k, v in checks.items():
    print(f'  {"✔" if v else "✘"} {k}')
GATE1 = all(checks.values())
print('GATE 1:', 'PASS' if GATE1 else 'FAIL', f'unexpected no-grad: {unexpected}' if unexpected else '')
save_result('gate1', {'curveA': curveA, 'curveB': curveB, 'evalA': evalA, 'evalB': evalB, 'hlB': hlB,
                      'ms_per_step': {'A_bs4': msA, 'B_bs20': msB}, 'params': n_params,
                      'unexpected_no_grad': unexpected, 'checks': {k: bool(v) for k, v in checks.items()},
                      'pass': bool(GATE1)})

check supports length 2 3


check supports length 2 3


check supports length 2 3


params=311,164   dead by design (final layer gconv/bn never reach the output): ['bn.7.weight', 'bn.7.bias', 'gconv.7.mlp.mlp.weight', 'gconv.7.mlp.mlp.bias']
A · 4 windows, full batch, 1000 steps (245 ms/step): 1:152.4  50:21.4  100:18.9  250:10.7  500:7.9  750:5.2  1000:4.6  | eval-mode 4.85
B · 100 windows, 50 epochs, bs 20 (1261 ms/step): 1:122.0  2:89.3  5:58.8  10:45.0  20:40.1  30:34.9  40:35.2  50:31.9  | eval-mode 30.31 | HL same windows 64.15
device memory (mps): 3.09 GiB
  ✔ A: final < 10% of step 1
  ✔ A: still improving in the 2nd half (last 100 steps < 80% of steps 401-500)
  ✔ A: eval-mode within 25% of train
  ✔ B: final < HL on same windows
  ✔ B: final < 40% of epoch 1
  ✔ every parameter except dead-by-design gets gradient (fp32 pass)
GATE 1: PASS 


## ⛔ GATE 1 — STOP

Paste the output of cells 08 and 09. Every line in 09's checklist should show ✔:
- **A**, 4 windows: the loss falls below 10% of where it started and is still improving in the second half. Eval-mode MAE matches train MAE, which confirms the BatchNorm running stats and the eval path are sound.
- **B**, 100 windows (your spec): a sanity curve that must end below HL and below 40% of epoch 1. It plateaus by design, because 100 windows give 859K targets for about 304K parameters.
- In a separate fp32 pass, every parameter gets a gradient except GWNet's final-layer `gconv`/`bn`. Those never reach the output, both here and in the original Graph WaveNet code.

Stage 2 (subset timing and VRAM at bs=64) is added after go-ahead.

## Stage 2 · Subset run and timing (Colab T4 ~10 min · M2 MPS ~40 min)

Runtime: Colab T4, or the M2 (plugged in, with heavy apps closed). In a fresh runtime run **00, 01, 07, 08** first. Re-run **07**, because `conglib.py` gained `evaluate()`.

- **10**: throughput and memory at the full 716 sensors with an effective batch of 64 (on the M2: micro-batches of 8, accumulated). It turns the measured speed into a full-epoch time, hours per run at 8/12/16 epochs, and the quarter-epochs that fit a ≤75-minute Colab-style session.
- **11**: GWNet on a 60-day subset for 5 epochs, with LargeST's settings. Validation uses windows inside days 50–59. It must beat HL, and a profile fitted on days 0–49, on those same windows.

In [5]:
# ── 10 · Stage 2a · throughput + memory at full N=716, effective batch 64, LargeST config ──
# CUDA: bs 64 with and without AMP. MPS (8 GB M2): micro-batches of 8 accumulated to 64, fp32 (+ fp16 for
# reference). Converts the measured speed into a full-epoch time, hours per run, and a <=75-min budget.
FORCE_STAGE2 = False
g1 = DRIVE_PROJ / 'results' / 'gate1.json'
if not FORCE_STAGE2:
    assert g1.exists(), 'No results/gate1.json: run cell 09 (Stage 1) first'
    _g1 = json.load(open(g1))
    assert _g1['pass'], ('Gate 1 FAILED — failing checks: '
                         f"{[k for k, v in _g1.get('checks', {}).items() if not v]}. Paste cell 09's output to Claude.")

def probe(amp, micro_bs, bs=64, n_warm=2, n_meas=8):
    conglib.set_seed(SEED)
    model = build_gwnet(adj, 'doubletransition', 1, 3, DEVICE, REPO)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=amp)
    rng = np.random.default_rng(SEED)
    reset_mem(DEVICE)
    try:
        first = None
        for i in range(n_warm + n_meas):
            if i == n_warm:
                sync(DEVICE)
                t0 = time.time()
            loss = train_step(model, opt, scaler, *data.batch(rng.choice(idx['train'], bs, replace=False)),
                              MEAN, STD, amp, micro_bs=micro_bs)
            first = loss if first is None else first
        sync(DEVICE)
        ms_train = 1000 * (time.time() - t0) / n_meas
        n_eval = EVAL_BS * 5
        t0 = time.time()
        evaluate(model, data, idx['val'][:n_eval], MEAN, STD, amp, bs=EVAL_BS)
        sync(DEVICE)
        return {'amp': amp, 'micro_bs': micro_bs, 'ms_train': ms_train, 'mem_gib': mem_gib(DEVICE),
                'ms_eval_per_window': 1000 * (time.time() - t0) / n_eval, 'first_loss': first}
    except Exception as e:
        if not is_oom(e):
            raise
        return {'amp': amp, 'micro_bs': micro_bs, 'oom': True}
    finally:
        del model, opt
        reset_mem(DEVICE)

if DEVICE.type == 'cuda':
    probes = [probe(True, None), probe(False, None)]
    if probes[0].get('oom'):
        probes.append(probe(True, 32))                       # 2 x 32, still an effective batch of 64
else:
    probes = [probe(False, MICRO_BS), probe(True, MICRO_BS)]  # fp32 is used; fp16 shown for reference
for r in probes:
    print(r if r.get('oom') else
          f"amp={str(r['amp']):5s} micro_bs={str(r['micro_bs']):4s} train {r['ms_train']:6.0f} ms/step (bs 64)  "
          f"eval {r['ms_eval_per_window']:5.1f} ms/window  memory {r['mem_gib']:.2f} GiB  first-step loss {r['first_loss']:.2f}")

best = next(r for r in probes if not r.get('oom') and r['amp'] == AMP)
BS = 64
steps_epoch = len(idx['train']) // BS
s_epoch = steps_epoch * best['ms_train'] / 1000
val_sub = idx['val'][::4]                                  # validation subsample used during training
s_val_sub = len(val_sub) * best['ms_eval_per_window'] / 1000
s_final_eval = (len(idx['val']) + len(idx['test'])) * best['ms_eval_per_window'] / 1000
s_quarter = s_epoch / 4 + s_val_sub + 3                    # + ~3 s checkpoint write
hours = {E: (4 * E * s_quarter + s_final_eval) / 3600 for E in (8, 12, 16)}
RUN_MIN = 75
n_quarters = int((RUN_MIN * 60 - s_final_eval - 60) // s_quarter)
BUDGET = {'device': DEVICE.type, 'bs': BS, 'micro_bs': best['micro_bs'], 'amp': best['amp'],
          's_per_full_epoch': s_epoch, 'hours_per_run': hours, 'quarters_in_75min': n_quarters}

print(f'\nfull epoch ({steps_epoch} steps x bs {BS}): {s_epoch / 60:.1f} min   val subsample: {s_val_sub:.0f} s   '
      f'final val+test eval: {s_final_eval / 60:.1f} min')
print(f'LargeST-style 100-epoch run ≈ {100 * (s_epoch + 4 * s_val_sub) / 3600:.1f} h on {DEVICE.type}')
print('hours per run (before thermal throttling): ' + '   '.join(f'{E} epochs ≈ {h:.1f} h' for E, h in hours.items()))
print(f'a <=75-min session fits {n_quarters} quarter-epochs = {n_quarters / 4:.1f} epochs')
if DEVICE.type == 'cuda' and n_quarters < 8:
    print(f'WARNING: only {n_quarters} quarter-epochs fit in {RUN_MIN} min — is this a T4 runtime?')
ok = [r for r in probes if not r.get('oom')]
if len(ok) >= 2:
    print(f"AMP vs fp32 first-step loss difference: {abs(ok[0]['first_loss'] - ok[1]['first_loss']):.3f} "
          '(same init, same batch)')

check supports length 2 3


check supports length 2 3


amp=False micro_bs=8    train   2959 ms/step (bs 64)  eval  21.4 ms/window  memory 3.06 GiB  first-step loss 146.65
amp=True  micro_bs=8    train   2724 ms/step (bs 64)  eval  15.1 ms/window  memory 2.10 GiB  first-step loss 146.65

full epoch (328 steps x bs 64): 16.2 min   val subsample: 38 s   final val+test eval: 5.0 min
LargeST-style 100-epoch run ≈ 31.1 h on mps
hours per run (before thermal throttling): 8 epochs ≈ 2.6 h   12 epochs ≈ 3.9 h   16 epochs ≈ 5.1 h
a <=75-min session fits 14 quarter-epochs = 3.5 epochs
AMP vs fp32 first-step loss difference: 0.000 (same init, same batch)


In [6]:
# ── 11 · Stage 2b · 60-day subset: 5 epochs, LargeST settings (effective batch 64) ──
# Train on windows whose targets fall in days 0-49; validate on windows lying entirely in days 50-59.
# Baselines on the same validation windows: HL (last value) and a profile fitted on days 0-49 only.
assert 'BUDGET' in globals(), 'Cell 10 has not completed in this runtime (it sets BS, BUDGET): run 09, then 10, then 11'
EPOCHS_S2, DAY = 5, STEPS_PER_DAY
tr = idx['train']
sub_train = tr[tr + HORIZON < 50 * DAY]
sub_val = tr[(tr - T_IN + 1 >= 50 * DAY) & (tr + HORIZON < 60 * DAY)]

conglib.set_seed(SEED)
model = build_gwnet(adj, 'doubletransition', 1, 3, DEVICE, REPO)            # dropout 0.3 (LargeST default)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=BUDGET['amp'])
rng = np.random.default_rng(SEED)
reset_mem(DEVICE)
log, peak = [], 0.0
for ep in range(1, EPOCHS_S2 + 1):
    order = sub_train[rng.permutation(len(sub_train))]
    sync(DEVICE)
    t0 = time.time()
    losses = [train_step(model, opt, scaler, *data.batch(order[i:i + BS]), MEAN, STD, BUDGET['amp'],
                         micro_bs=BUDGET['micro_bs'])
              for i in range(0, len(order) // BS * BS, BS)]
    sync(DEVICE)
    t_train = time.time() - t0
    peak = max(peak, mem_gib(DEVICE))
    val = evaluate(model, data, sub_val, MEAN, STD, BUDGET['amp'], bs=EVAL_BS)
    log.append({'epoch': ep, 'train_mae': float(np.mean(losses)), 'val_mae': float(val[0].mean()),
                's_train': t_train, 's_per_step': t_train / len(losses)})
    print(f"epoch {ep}: train MAE {log[-1]['train_mae']:.2f}  subset-val MAE {log[-1]['val_mae']:.2f}  "
          f"{t_train:.0f} s ({1000 * t_train / len(losses):.0f} ms/step)", flush=True)

slot = (TIMES.hour * 4 + TIMES.minute // 15).to_numpy()
dtype3 = np.select([TIMES.dayofweek < 5, TIMES.dayofweek == 5], [0, 1], 2)
Fsub = np.where(F[:50 * DAY] == 0, np.nan, F[:50 * DAY])
prof50 = np.full((3, STEPS_PER_DAY, N), np.nan, np.float32)
for d in range(3):
    for k in range(STEPS_PER_DAY):
        prof50[d, k] = np.nanmedian(Fsub[(dtype3[:50 * DAY] == d) & (slot[:50 * DAY] == k)], axis=0)
prof50 = np.nan_to_num(prof50, nan=float(np.nanmedian(Fsub)))

def baseline_mae(pred_fn):
    out = []
    for h in range(1, HORIZON + 1):
        y = F[sub_val + h]
        m = y != 0
        out.append(np.abs(pred_fn(sub_val, h)[m] - y[m]).mean())
    return np.array(out)

gw = evaluate(model, data, sub_val, MEAN, STD, BUDGET['amp'], bs=EVAL_BS)[0]
hl_s = baseline_mae(lambda ix, h: F[ix])
pr_s = baseline_mae(lambda ix, h: prof50[dtype3[ix + h], slot[ix + h]])
tab = pd.DataFrame({'GWNet 5ep': gw, 'HL': hl_s, 'profile(d0-49)': pr_s}, index=range(1, HORIZON + 1)).T
tab['avg'] = tab.mean(axis=1)
print(f'\nsubset-val MAE by horizon ({len(sub_val)} windows, days 50-59):')
print(tab[[1, 2, 3, 4, 6, 8, 12, 'avg']].round(2).to_string())

if DEVICE.type == 'cuda':
    mem_limit = 14.0                                        # T4 has 15 GiB
elif DEVICE.type == 'mps':
    mem_limit = 0.9 * torch.mps.recommended_max_memory() / 2**30
else:
    mem_limit = float('inf')
steady = log[1:] or log                                    # epoch 1 includes kernel warm-up
throttle = log[-1]['s_per_step'] / steady[0]['s_per_step']
checks = {
    f'running on a GPU ({DEVICE.type})': DEVICE.type in ('cuda', 'mps'),
    f'device memory {peak:.2f} GiB within limit ({mem_limit:.1f} GiB)': peak < mem_limit,
    'train loss descends (epoch 5 < 0.8 × epoch 1)': log[-1]['train_mae'] < 0.8 * log[0]['train_mae'],
    'beats HL on subset-val (avg MAE)': tab.loc['GWNet 5ep', 'avg'] < tab.loc['HL', 'avg'],
    'beats the profile at horizons 1-3 (profile ignores current state)':
        bool((tab.loc['GWNet 5ep', [1, 2, 3]] < tab.loc['profile(d0-49)', [1, 2, 3]]).all()),
}
for k, v in checks.items():
    print(f'  {"✔" if v else "✘"} {k}')
GATE2 = all(checks.values())
print('GATE 2:', 'PASS' if GATE2 else 'FAIL')
print(f"profile vs GWNet at avg (informational, 5 epochs only): {tab.loc['profile(d0-49)', 'avg']:.2f} vs "
      f"{tab.loc['GWNet 5ep', 'avg']:.2f}")
ms_steady = 1000 * np.mean([r['s_per_step'] for r in steady[-2:]])
epoch_min = steps_epoch * ms_steady / 60000
print(f'step time last epoch vs epoch 2: ×{throttle:.2f} (thermal throttling if clearly > 1.1)')
print(f'measured, sustained: {ms_steady:.0f} ms/step → full epoch ≈ {epoch_min:.1f} min → a 12-epoch run ≈ '
      f'{(12 * (epoch_min * 60 + 4 * s_val_sub) + s_final_eval) / 3600:.1f} h on {DEVICE.type}')
save_result('gate2', {'probes': probes, 'budget': BUDGET, 'subset_log': log, 'peak_gib': peak,
                      'throttle_ratio': throttle, 'sustained_ms_per_step': ms_steady,
                      'subset_val_mae': tab.round(3).to_dict(), 'checks': {k: bool(v) for k, v in checks.items()},
                      'pass': bool(GATE2)})
del model, opt

check supports length 2 3


epoch 1: train MAE 58.96  subset-val MAE 44.50  216 s (2926 ms/step)


epoch 2: train MAE 39.62  subset-val MAE 36.64  241 s (3257 ms/step)


epoch 3: train MAE 37.45  subset-val MAE 36.58  255 s (3451 ms/step)


epoch 4: train MAE 36.60  subset-val MAE 36.96  252 s (3411 ms/step)


epoch 5: train MAE 34.81  subset-val MAE 37.82  234 s (3162 ms/step)



subset-val MAE by horizon (937 windows, days 50-59):
                    1      2      3      4      6      8      12    avg
GWNet 5ep       19.36  23.09  26.82  30.32  37.62  43.57   52.39  37.82
HL              19.15  28.37  37.33  46.08  64.30  81.08  112.26  67.38
profile(d0-49)  20.11  20.16  20.22  20.27  20.40  20.52   20.69  20.42
  ✔ running on a GPU (mps)
  ✔ device memory 3.06 GiB within limit (4.8 GiB)
  ✔ train loss descends (epoch 5 < 0.8 × epoch 1)
  ✔ beats HL on subset-val (avg MAE)
  ✘ beats the profile at horizons 1-3 (profile ignores current state)
GATE 2: FAIL
profile vs GWNet at avg (informational, 5 epochs only): 20.42 vs 37.82
step time last epoch vs epoch 2: ×0.97 (thermal throttling if clearly > 1.1)
measured, sustained: 3287 ms/step → full epoch ≈ 18.0 min → a 12-epoch run ≈ 4.2 h on mps


## ⛔ GATE 2 — STOP

Paste the output of cells 10 and 11. Report back:
- ms/step and peak VRAM at bs 64, with and without AMP, and the AMP-vs-fp32 first-step loss gap, which should be tiny;
- the extrapolated full-epoch time and the proposed per-run budget, which is what we give up versus LargeST's 100 epochs;
- the ✔/✘ checklist: GPU present, memory fits, loss descends, the model beats HL on average and the profile at horizons 1–3.

Stage 3 comes after go-ahead. It adds `train.py` under `nohup` with Drive checkpoints every quarter-epoch, a resume test, and the first ~25% of the R0 budget.

## Stage 3 · Resumable background training, Gate 3 = first 25% of R0

Run **00, 01, 07, 08** first, then **12 → 13 → 14**. Cell **15** can be re-run at any time to check progress.

- **12** writes `train.py`, the resumable trainer. Every quarter-epoch it writes checkpoints (`last.pt`, and `best.pt` when validation improves) and a history file. The learning rate goes warmup → constant 1e-3 (LargeST's) → cosine decay over the last 30%, so R0's total length can still be changed after Gate 3.
- **13** is the resume self-test: an interrupted-then-resumed run must reproduce an uninterrupted one.
- **14** launches R0 in the background. On the Mac it runs under `caffeinate`, so the Mac doesn't sleep, and it survives this kernel. For Gate 3 it pauses after 12 quarters (3 epochs, about 1 h on the M2). Re-run 14 with `STOP_AFTER = 0` to continue.
- **15** shows status, the learning curve, the ETA, and GWNet vs HL and the profile, per horizon, on the same validation windows.

In [5]:
%%writefile {WORK}/cong/train.py
# ── 12 · train.py — resumable background trainer (checkpoints every quarter-epoch) ──
"""Resumable GWNet training for the R/A runs — launched in the background (nohup / caffeinate) by cell 14.

Every quarter-epoch: evaluate on a validation subsample, write <proj>/ckpt/<run>/last.pt (and best.pt when
the validation MAE improves) atomically, and append to <proj>/results/<run>_history.json. Re-running the
same command resumes from last.pt; each epoch's shuffle is derived from (seed, epoch), so a resumed run
visits exactly the windows an uninterrupted run would.

LR: linear warmup -> constant (LargeST's 1e-3) -> cosine decay over the last `decay_frac` of the run.
The total length can therefore still be changed on resume as long as the decay has not started.
"""
import argparse
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path(__file__).resolve().parent))
import conglib  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument('--run', required=True)
ap.add_argument('--data', required=True, help='LargeST data/sd folder')
ap.add_argument('--proj', required=True, help='project results folder (ckpt/, results/ live here)')
ap.add_argument('--repo', required=True, help='LargeST repo (for src.models.gwnet)')
ap.add_argument('--epochs', type=float, default=12)
ap.add_argument('--stop-after-quarters', type=int, default=0, help='pause after this many quarters (0 = run to the end)')
ap.add_argument('--lr', type=float, default=1e-3)
ap.add_argument('--wd', type=float, default=1e-4)
ap.add_argument('--dropout', type=float, default=0.3)
ap.add_argument('--bs', type=int, default=64)
ap.add_argument('--adj-type', default='doubletransition')
ap.add_argument('--adp', type=int, default=1)
ap.add_argument('--warmup-quarters', type=float, default=1)
ap.add_argument('--decay-frac', type=float, default=0.3)
ap.add_argument('--val-stride', type=int, default=4)
ap.add_argument('--seed', type=int, default=2023)
ap.add_argument('--task', choices=['flow', 'mt'], default='flow', help='mt = flow + congestion-class head')
ap.add_argument('--lam', type=float, default=15.0, help='weight of the class-CE term (task mt)')
ap.add_argument('--inputs', choices=['R0', 'R2', 'R3', 'R4', 'R5', 'R6'], default='R0', help='input feature level (features.py)')
ap.add_argument('--debug-steps', type=int, default=0, help='tests only: cap steps per quarter')
ap.add_argument('--skip-final', action='store_true', help='tests only: no full val/test evaluation at the end')
a = ap.parse_args()

T_IN, HORIZON = conglib.T_IN, conglib.HORIZON
ckpt_dir = Path(a.proj) / 'ckpt' / a.run
res_dir = Path(a.proj) / 'results'
ckpt_dir.mkdir(parents=True, exist_ok=True)
res_dir.mkdir(parents=True, exist_ok=True)
(ckpt_dir / 'pid').write_text(str(os.getpid()))

def log(msg):
    print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] {msg}', flush=True)

# ── data: R0 = LargeST's 3 input channels; targets = raw flow, kept separate from the inputs ──
DEVICE = conglib.pick_device()
AMP = DEVICE.type == 'cuda'
MICRO_BS = 8 if DEVICE.type == 'mps' else None
EVAL_BS = 16 if DEVICE.type == 'mps' else 64
with np.load(Path(a.data) / '2019' / 'his.npz') as z:
    X = z['data'].astype(np.float32)
    MEAN, STD = float(z['mean']), float(z['std'])
F = pd.read_hdf(Path(a.data) / 'sd_his_2019.h5', key='t').to_numpy(np.float32)
assert np.abs(X[..., 0] * STD + MEAN - F).max() < 1e-2, 'input channel 0 is not the normalised target'
idx = {k: np.load(Path(a.data) / '2019' / f'idx_{k}.npy') for k in ('train', 'val', 'test')}
TRAIN_END = int(idx['val'][0] - T_IN)
MT = a.task == 'mt'
if MT:                                                # hourly occupancy classes -> 15-min grid (see Stage L)
    LAB = np.repeat(np.load(Path(a.proj) / 'factors' / 'cong_label.npz')['label_hour'], 4, axis=0)
    share = np.array([(LAB[:TRAIN_END] == k).sum() for k in range(3)], float)
    share /= share.sum()
    CLASS_W = torch.tensor(1 / np.sqrt(share) / (1 / np.sqrt(share)).mean(), dtype=torch.float32, device=DEVICE)
if a.inputs == 'R0':
    data = conglib.Windows(X, F, DEVICE, Yc=LAB if MT else None)
    IN_NAMES = ['flow', 'tod', 'dow/7']
else:
    import features
    fin = features.assemble(a.inputs, Path(a.data), Path(a.proj) / 'factors')
    ok = ~fin['miss']                                 # channel 0 = normalised flow; untouched where observed
    assert np.abs(fin['dyn'][..., 0][ok].astype(np.float32) * STD + MEAN - F[ok]).max() < 0.6, 'flow channel != target'
    data = conglib.Windows(fin['dyn'], F, DEVICE, x_dtype=torch.float16, Yc=LAB if MT else None, time=fin['time'],
                           static=fin['static'], wx=fin['wx'], wx_w=fin['wx_w'])
    IN_NAMES = fin['names']
    del fin
del X
adj = np.load(Path(a.data) / 'sd_rn_adj.npy')
val_sub = idx['val'][::a.val_stride]

# ── schedule ──────────────────────────────────────────────────────────────
total_q = int(round(4 * a.epochs))
spq = (len(idx['train']) // 4) // a.bs                          # optimiser steps per quarter
if a.debug_steps:
    spq = min(spq, a.debug_steps)
total_steps = total_q * spq
warm = int(a.warmup_quarters * spq)
decay_start = int((1 - a.decay_frac) * total_steps)

def lr_at(step):
    if step < warm:
        return a.lr * (0.1 + 0.9 * step / max(warm, 1))
    if step < decay_start:
        return a.lr
    return a.lr * 0.5 * (1 + math.cos(math.pi * (step - decay_start) / max(total_steps - decay_start, 1)))

# ── model / state (fresh or resumed) ──────────────────────────────────────
conglib.set_seed(a.seed)
IN_DIM = data.n_channels if a.inputs != 'R0' else 3
model = conglib.build_gwnet(adj, a.adj_type, a.adp, input_dim=IN_DIM, device=DEVICE, repo=a.repo, dropout=a.dropout)
if MT:
    model = conglib.GWNetMT(model).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=a.lr, weight_decay=a.wd)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)
state = {'q': 0, 'step': 0, 'best_val': float('inf'), 'history': [], 'args': vars(a)}
last = ckpt_dir / 'last.pt'
if last.exists():
    ck = torch.load(last, map_location='cpu', weights_only=False)
    old = ck['state']['args']
    for k in ('lr', 'wd', 'dropout', 'bs', 'adj_type', 'adp', 'seed', 'warmup_quarters', 'decay_frac', 'val_stride', 'task', 'lam', 'inputs'):
        assert old.get(k, vars(a)[k]) == vars(a)[k], f'--{k.replace("_", "-")} differs from the checkpoint ({old[k]} vs {vars(a)[k]})'
    old_decay = int((1 - old['decay_frac']) * int(round(4 * old['epochs'])) * spq)
    assert old['epochs'] == a.epochs or ck['state']['step'] < min(old_decay, decay_start), \
        'cannot change --epochs after the LR decay has started'
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    scaler.load_state_dict(ck['scaler'])
    torch.set_rng_state(ck['rng_cpu'])
    if DEVICE.type == 'mps' and ck.get('rng_mps') is not None:
        torch.mps.set_rng_state(ck['rng_mps'])
    if DEVICE.type == 'cuda' and ck.get('rng_cuda') is not None:
        torch.cuda.set_rng_state(ck['rng_cuda'])
    state = ck['state']
    state['args'] = vars(a)
    log(f'resumed {a.run} at quarter {state["q"]}/{total_q}, step {state["step"]}, best val {state["best_val"]:.3f}')
else:
    log(f'new run {a.run} ({a.task}, inputs {a.inputs}: {IN_DIM} channels): {total_q} quarters x {spq} steps (bs {a.bs}, micro {MICRO_BS}), device {DEVICE}, '
        f'params {sum(p.numel() for p in model.parameters()):,}, warmup {warm} steps, decay from step {decay_start}'
        + (f', lam {a.lam}, class weights {CLASS_W.cpu().numpy().round(3).tolist()}' if MT else ''))

def save(path, extra=None):
    tmp = path.with_suffix('.tmp')
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(),
                'rng_cpu': torch.get_rng_state(),
                'rng_mps': torch.mps.get_rng_state() if DEVICE.type == 'mps' else None,
                'rng_cuda': torch.cuda.get_rng_state() if DEVICE.type == 'cuda' else None,
                'state': state, **(extra or {})}, tmp)
    os.replace(tmp, path)

# ── train ─────────────────────────────────────────────────────────────────
while state['q'] < total_q:
    if a.stop_after_quarters and state['q'] >= a.stop_after_quarters:
        log(f'paused after {state["q"]} quarters (--stop-after-quarters); re-run without it to continue')
        sys.exit(0)
    q = state['q']
    epoch, part = divmod(q, 4)
    perm = np.random.default_rng(a.seed + epoch).permutation(idx['train'])
    chunk = np.array_split(perm, 4)[part][:spq * a.bs].reshape(spq, a.bs)
    t0 = time.time()
    losses, ces = [], []
    for b in chunk:
        for g in opt.param_groups:
            g['lr'] = lr_at(state['step'])
        if MT:
            l, c = conglib.train_step_mt(model, opt, scaler, *data.batch(b), MEAN, STD, AMP, a.lam, CLASS_W, micro_bs=MICRO_BS)
            losses.append(l)
            ces.append(c)
        else:
            losses.append(conglib.train_step(model, opt, scaler, *data.batch(b), MEAN, STD, AMP, micro_bs=MICRO_BS))
        state['step'] += 1
    conglib.sync(DEVICE)
    t_train = time.time() - t0
    if MT:
        val, vconf = conglib.evaluate_mt(model, data, val_sub, MEAN, STD, AMP, bs=EVAL_BS)
        _, _, vf1, vmacro = conglib.class_scores(vconf)
    else:
        val = conglib.evaluate(model, data, val_sub, MEAN, STD, AMP, bs=EVAL_BS)
    state['q'] = q + 1
    rec = {'q': q + 1, 'epoch': (q + 1) / 4, 'step': state['step'], 'lr': lr_at(state['step'] - 1),
           'train_mae': float(np.mean(losses)), 'val_mae': float(val[0].mean()),
           'val_mae_h': [round(float(v), 3) for v in val[0]], 's_train': t_train,
           'ms_per_step': 1000 * t_train / len(chunk), 's_total': time.time() - t0}
    if MT:
        rec.update({'train_ce': float(np.mean(ces)), 'val_macro_f1': float(vmacro.mean()),
                    'val_f1_congested': float(vf1[:, 2].mean()), 'val_macro_f1_h': [round(float(v), 4) for v in vmacro]})
    state['history'].append(rec)
    improved = rec['val_mae'] < state['best_val']
    if improved:
        state['best_val'] = rec['val_mae']
        save(ckpt_dir / 'best.pt')
    save(last)
    (res_dir / f'{a.run}_history.json').write_text(
        json.dumps({'total_q': total_q, 'args': vars(a), 'history': state['history']}, indent=0))
    log(f"q {q + 1:3d}/{total_q} (ep {rec['epoch']:5.2f})  train {rec['train_mae']:7.3f}  val-sub {rec['val_mae']:7.3f}"
        f"{' *' if improved else '  '}  lr {rec['lr']:.2e}  {rec['ms_per_step']:.0f} ms/step  {rec['s_total']:.0f} s"
        + (f"  | ce {rec['train_ce']:.3f}  val macro-F1 {rec['val_macro_f1']:.3f}  F1(congested) {rec['val_f1_congested']:.3f}" if MT else ''))

# ── final: best checkpoint on the full val and test sets (LargeST protocol) ──
if a.skip_final:
    log('done (final evaluation skipped)')
    sys.exit(0)
best = torch.load(ckpt_dir / 'best.pt', map_location='cpu', weights_only=False)
model.load_state_dict(best['model'])
final = {}
for split in ('val', 'test'):
    ix = idx[split][::50] if a.debug_steps else idx[split]
    if MT:
        m, conf = conglib.evaluate_mt(model, data, ix, MEAN, STD, AMP, bs=EVAL_BS)
    else:
        m = conglib.evaluate(model, data, ix, MEAN, STD, AMP, bs=EVAL_BS)
    final[split] = {'MAE': m[0].tolist(), 'RMSE': m[1].tolist(), 'MAPE': m[2].tolist(),
                    'avg': [float(m[0].mean()), float(m[1].mean()), float(m[2].mean())]}
    if MT:
        prec, recl, f1, macro = conglib.class_scores(conf)
        final[split]['cls'] = {'macro_f1_h': macro.tolist(), 'macro_f1': float(macro.mean()),
                               'precision': prec.mean(0).tolist(), 'recall': recl.mean(0).tolist(), 'f1': f1.mean(0).tolist(),
                               'confusion_h': conf.tolist(), 'classes': ['free', 'heavy', 'congested']}
        log(f"{split}: macro-F1 {macro.mean():.3f} (h1 {macro[0]:.3f}, h12 {macro[11]:.3f})  recall free/heavy/congested "
            f"{' / '.join(f'{v:.2f}' for v in recl.mean(0))}  precision {' / '.join(f'{v:.2f}' for v in prec.mean(0))}")
    log(f"{split}: avg MAE {final[split]['avg'][0]:.3f}  RMSE {final[split]['avg'][1]:.3f}  MAPE {final[split]['avg'][2]:.4f}"
        f"  | MAE h3 {m[0][2]:.2f} h6 {m[0][5]:.2f} h12 {m[0][11]:.2f}")
(res_dir / f'{a.run}_final.json').write_text(json.dumps({'final': final, 'best_val_sub': state['best_val'],
                                                          'args': vars(a)}, indent=1))
log('done')

Writing /Users/lavanyapareek/Documents/Capstone_Post_Joining/work/cong/train.py


In [6]:
# ── 13 · Resume self-test (~2 min): interrupted + resumed must reproduce an uninterrupted run ──
TRAIN = WORK / 'cong' / 'train.py'
test_proj = WORK / '_resume_test'
shutil.rmtree(test_proj, ignore_errors=True)
base = [sys.executable, str(TRAIN), '--data', str(DATA), '--proj', str(test_proj), '--repo', str(REPO),
        '--epochs', '0.75', '--debug-steps', '2', '--val-stride', '50', '--skip-final']

def run_train(*extra):
    r = subprocess.run(base + list(extra), capture_output=True, text=True)
    assert r.returncode == 0, (r.stdout + r.stderr)[-3000:]
    return r.stdout

run_train('--run', 'straight')                                   # 3 quarters in one go
run_train('--run', 'split', '--stop-after-quarters', '1')        # 1 quarter, then stop ...
resumed = run_train('--run', 'split')                            # ... and resume for the other 2
h1 = json.load(open(test_proj / 'results' / 'straight_history.json'))['history']
h2 = json.load(open(test_proj / 'results' / 'split_history.json'))['history']
print(''.join(l + '\n' for l in resumed.splitlines() if 'resumed' in l or 'q ' in l), end='')
same_path = [r['step'] for r in h1] == [r['step'] for r in h2] and [r['lr'] for r in h1] == [r['lr'] for r in h2]
diff = max(abs(x[k] - y[k]) for x, y in zip(h1, h2) for k in ('train_mae', 'val_mae'))
RESUME_OK = len(h1) == len(h2) == 3 and same_path and diff < 1e-3
print(f'steps/LR identical: {same_path}   max |straight − resumed| (train & val MAE): {diff:.2e}   '
      f'RESUME TEST: {"PASS" if RESUME_OK else "FAIL"}')
save_result('resume_test', {'pass': bool(RESUME_OK), 'max_diff': diff, 'same_steps_lr': bool(same_path)})
shutil.rmtree(test_proj)

[2026-09-13 19:39:52] resumed split at quarter 1/3, step 2, best val 147.024
[2026-09-13 19:40:01] q   2/3 (ep  0.50)  train 134.167  val-sub 114.514 *  lr 1.00e-03  3289 ms/step  9 s
[2026-09-13 19:40:10] q   3/3 (ep  0.75)  train 106.546  val-sub  83.489 *  lr 5.00e-04  3047 ms/step  9 s
steps/LR identical: True   max |straight − resumed| (train & val MAE): 0.00e+00   RESUME TEST: PASS


In [2]:
# ── 14 · Launch or resume a run in the background (survives this kernel; re-run = resume) ──
# Run configs: every run is 16 epochs on R0's inputs unless noted; the A-runs are the GWNet graph 2x2 ablation.
RUNS = {'R0': {},                                              # road graph + adaptive adjacency (done)
        'A3': {'adj-type': 'none', 'adp': 0},                  # no graph at all: per-sensor temporal model
        'A1': {'adp': 0},                                      # road graph only
        'A2': {'adj-type': 'none', 'adp': 1},                  # adaptive adjacency only
        'R1': {'task': 'mt', 'lam': 15},                       # R0 + congestion-class head (Stage L labels)
        **{r: {'task': 'mt', 'lam': 15, 'inputs': r} for r in ('R2', 'R3', 'R4', 'R5', 'R6')}}   # cumulative feature rows
RUN, EPOCHS, STOP_AFTER = 'A3', 16, 0        # STOP_AFTER > 0 pauses after that many quarters
_rt = DRIVE_PROJ / 'results' / 'resume_test.json'
assert (_rt.exists() and json.load(open(_rt))['pass']) or (DRIVE_PROJ / 'ckpt' / RUN / 'last.pt').exists(), \
    'run cell 13 (resume self-test) first'
TRAIN = WORK / 'cong' / 'train.py'
(DRIVE_PROJ / 'logs').mkdir(exist_ok=True)
logf = DRIVE_PROJ / 'logs' / f'{RUN}.log'

def pid_alive(run):
    pidf = DRIVE_PROJ / 'ckpt' / run / 'pid'
    try:
        os.kill(int(pidf.read_text()), 0)
        return True
    except (FileNotFoundError, ProcessLookupError, ValueError):
        return False

if pid_alive(RUN):
    print(f'{RUN} is already running — check it with cell 15')
else:
    cmd = [sys.executable, str(TRAIN), '--run', RUN, '--data', str(DATA), '--proj', str(DRIVE_PROJ),
           '--repo', str(REPO), '--epochs', str(EPOCHS)] + (['--stop-after-quarters', str(STOP_AFTER)] if STOP_AFTER else [])
    cmd += [x for k, v in RUNS[RUN].items() for x in (f'--{k}', str(v))]
    if shutil.which('caffeinate'):
        cmd = ['caffeinate', '-dimsu'] + cmd                    # macOS: no sleep while training
    proc = subprocess.Popen(cmd, stdout=open(logf, 'a'), stderr=subprocess.STDOUT, start_new_session=True,
                            cwd=str(WORK / 'cong'))
    print(f'launched {RUN} {RUNS[RUN]} (pid {proc.pid}): {EPOCHS} epochs' + (f', pausing after {STOP_AFTER} quarters' if STOP_AFTER else '')
          + f'\nlog: {logf}')

launched A3 {'adj-type': 'none', 'adp': 0} (pid 55260): 16 epochs
log: /Users/lavanyapareek/Documents/Capstone_Post_Joining/data/traffic/congestion/logs/A3.log


In [3]:
# ── 15 · Run status: log tail, learning curve, ETA, per-horizon vs baselines on the same val windows ──
RUN = 'A3'                                     # which run to inspect (R0 is finished: results/R0_final.json)
logf = DRIVE_PROJ / 'logs' / f'{RUN}.log'
hf = DRIVE_PROJ / 'results' / f'{RUN}_history.json'
pidf = DRIVE_PROJ / 'ckpt' / RUN / 'pid'
try:
    os.kill(int(pidf.read_text()), 0)
    status = 'RUNNING'
except (FileNotFoundError, ProcessLookupError, ValueError):
    status = 'not running'
print(f'{RUN}: {status}')
print(''.join(open(logf).readlines()[-5:]) if logf.exists() else 'no log yet')

if hf.exists():
    J = json.load(open(hf))
    H = pd.DataFrame(J['history'])
    print(H[['q', 'epoch', 'train_mae', 'val_mae', 'lr', 'ms_per_step', 's_total']].round(3).to_string(index=False))
    left = J['total_q'] - len(H)
    print(f"\nprogress {len(H)}/{J['total_q']} quarters; ETA for the rest ≈ {left * H['s_total'].median() / 3600:.1f} h "
          f"+ final val/test eval")

    vs = idx['val'][::J['args']['val_stride']]               # the validation subsample train.py uses
    slot = (TIMES.hour * 4 + TIMES.minute // 15).to_numpy()
    dtype3 = np.select([TIMES.dayofweek < 5, TIMES.dayofweek == 5], [0, 1], 2)
    PROFILE = load_factor('profile_median_train')['profile']

    def base_mae(pred_fn):
        out = []
        for h in range(1, HORIZON + 1):
            y = F[vs + h]
            m = y != 0
            out.append(np.abs(pred_fn(vs, h)[m] - y[m]).mean())
        return out

    tab = pd.DataFrame({f"GWNet q{int(H['q'].iloc[-1])}": H['val_mae_h'].iloc[-1],
                        'best so far': H.loc[H['val_mae'].idxmin(), 'val_mae_h'],
                        'HL': base_mae(lambda ix, h: F[ix]),
                        'HA profile (train)': base_mae(lambda ix, h: PROFILE[dtype3[ix + h], slot[ix + h]])},
                       index=range(1, HORIZON + 1)).T
    tab['avg'] = tab.mean(axis=1)
    print(f'\nval-subsample MAE by horizon ({len(vs)} windows):')
    print(tab[[1, 2, 3, 4, 6, 8, 12, 'avg']].round(2).to_string())
    print('reference: your LSTM final val MAE 26.15; paper GWNet test 17.74 (val not published)')

    if len(H) >= 6:                                            # trend of the constant-LR phase (rough)
        from scipy.optimize import curve_fit
        qq, vv = H['q'].to_numpy(float)[1:], H['val_mae'].to_numpy()[1:]
        (c, b, k), _ = curve_fit(lambda q, c, b, k: c + b * q ** (-k), qq, vv, p0=(vv.min() * 0.6, vv[0], 0.5),
                                 bounds=([0, 0, 0.05], [vv.min(), np.inf, 3]), maxfev=20000)
        loglin = np.polyfit(np.log(qq), vv, 1)
        for Q in sorted({J['total_q'], 48, 64}):
            print(f'trend at quarter {Q} ({Q / 4:g} epochs), constant LR: power law {c + b * Q ** (-k):.2f} '
                  f'… log-linear {np.polyval(loglin, np.log(Q)):.2f}')
        print('(extrapolations from a still-falling curve are rough; the cosine-decay phase usually adds a further drop)')

R0: not running
[2026-09-14 20:01:06] q  63/64 (ep 15.75)  train  18.408  val-sub  18.545 *  lr 6.84e-06  3546 ms/step  327 s
[2026-09-14 20:06:39] q  64/64 (ep 16.00)  train  18.473  val-sub  18.584    lr 9.95e-10  3590 ms/step  332 s
[2026-09-14 20:09:09] val: avg MAE 18.596  RMSE 29.574  MAPE 0.1138  | MAE h3 16.50 h6 18.95 h12 21.77
[2026-09-14 20:11:33] test: avg MAE 20.347  RMSE 33.479  MAPE 0.1342  | MAE h3 16.92 h6 20.57 h12 25.35
[2026-09-14 20:11:33] done

 q  epoch  train_mae  val_mae    lr  ms_per_step  s_total
 1   0.25     75.217   43.376 0.001     2926.782  271.284
 2   0.50     41.513   36.528 0.001     3583.626  328.053
 3   0.75     38.161   35.753 0.001     3903.019  355.929
 4   1.00     35.578   34.882 0.001     4437.055  403.740
 5   1.25     34.001   32.829 0.001     3850.479  353.501
 6   1.50     32.364   30.994 0.001     3761.887  344.997
 7   1.75     32.137   30.845 0.001     3756.572  344.186
 8   2.00     31.221   29.508 0.001     3625.845  332.928
 9   2.

trend at quarter 48 (12 epochs), constant LR: power law 20.14 … log-linear 19.92
trend at quarter 64 (16 epochs), constant LR: power law 18.98 … log-linear 18.38
(extrapolations from a still-falling curve are rough; the cosine-decay phase usually adds a further drop)


## ⛔ GATE 3 — STOP

Once R0 has paused after 12 quarters, run **15** and report:
- the validation-subsample learning curve over 3 epochs, and whether it is still falling;
- GWNet vs HL and the train-fitted profile, per horizon, on the same windows. At horizons 1–3 it should now beat both clearly; this settles the Gate 2 question;
- the trend-fit projection, compared with your LSTM (val 26.15) and the paper's GWNet (17.74);
- a decision on R0's total length (12 or 16 epochs; set `EPOCHS` in cell 14 and re-run it with `STOP_AFTER = 0`).

## Stage L · Congestion label from PeMS occupancy (CPU)

Run **00, 01**, then **L1–L6**. Inputs are the `pems_filter.py` outputs in `congestion/pems/`: hourly occupancy for all of 2019, 15-min occupancy for Nov 4–10, and station metadata.

- **Label.** Each sensor gets a critical occupancy o_c: the occupancy at which its flow peaks. It's fitted on *training hours only*. Classes per hour are **free** (occ < 0.7·o_c), **heavy** (0.7–1.0·o_c) and **congested** (≥ o_c). Each 15-min step takes its hour's class. Hours with no record, no occupancy, or under 50% observed are **ignored** (−1).
- **Checks.** Class shares per split; what the hourly resolution loses against the 15-min week; whether congestion spreads upstream the way real queues do; and how a flow-only label compares, especially on holidays.

In [4]:
# ── L1 · Load the PeMS extracts and re-check them against LargeST ────────────
PEMS = DRIVE_PROJ / 'pems'
_h = np.load(PEMS / 'pems_sd_2019_hour.npz')
_q = np.load(PEMS / 'pems_sd_2019_15min.npz')
assert (_h['ids'] == meta['ID'].to_numpy()).all() and (_q['ids'] == meta['ID'].to_numpy()).all(), 'sensor order differs'
TH = pd.date_range('2019-01-01', periods=8760, freq='h')
HOCC, HFLOW, HPO, HN = _h['occ'].astype(np.float32), _h['flow'], _h['pct_obs'], _h['nrec']
H_OK = (HN > 0) & (HPO >= 50) & np.isfinite(HOCC)                  # observed well enough to label
TRAIN_END_H = TRAIN_END // 4                                        # hours entirely inside the training window
Q_OCC, Q_PO, Q_N = _q['occ'].astype(np.float32), _q['pct_obs'], _q['nrec']

Fh = F.reshape(8760, 4, N)
both = (Fh > 0).all(1) & (HN > 0)
print(f'hourly PeMS vs LargeST flow: corr {np.corrcoef(HFLOW[both], Fh.mean(1)[both])[0, 1]:.5f} over {both.sum():,} sensor-hours')
print(f'labelable sensor-hours (record, occupancy, ≥50% observed): {H_OK.mean():.3f}   '
      f'train hours: {TRAIN_END_H} ({TH[0]:%b %d} → {TH[TRAIN_END_H - 1]:%b %d %H:%M})')
print(f'hourly occupancy percentiles (labelable): p50 {np.percentile(HOCC[H_OK], 50):.3f}  p90 {np.percentile(HOCC[H_OK], 90):.3f}  '
      f'p99 {np.percentile(HOCC[H_OK], 99):.3f}')

hourly PeMS vs LargeST flow: corr 0.99998 over 6,119,193 sensor-hours
labelable sensor-hours (record, occupancy, ≥50% observed): 0.882   train hours: 5252 (Jan 01 → Aug 07 19:00)
hourly occupancy percentiles (labelable): p50 0.053  p90 0.119  p99 0.342


In [5]:
# ── L2 · Critical occupancy per sensor, fitted on training hours only ──────────
# o_c = occupancy where the upper envelope (p95) of hourly flow peaks on the flow–occupancy curve.
# A sensor whose envelope never drops after the peak has no observed congested branch → network median.
OBINS = np.arange(0, 0.61, 0.01)
OC = np.full(N, np.nan, np.float32)
OC_SRC = np.zeros(N, np.int8)              # 0 = too little data, 1 = fitted, 2 = no congested branch → median
for s in range(N):
    ok = H_OK[:TRAIN_END_H, s]
    o, q = HOCC[:TRAIN_END_H, s][ok], HFLOW[:TRAIN_END_H, s][ok]
    if ok.sum() < 500:
        continue
    b = np.clip(np.digitize(o, OBINS) - 1, 0, len(OBINS) - 2)
    env = np.array([np.percentile(q[b == k], 95) if (b == k).sum() >= 15 else np.nan for k in range(len(OBINS) - 1)])
    sm = pd.Series(env).rolling(3, center=True, min_periods=1).mean().to_numpy()
    k = int(np.nanargmax(sm))
    after = sm[k + 1:]
    if np.isfinite(after).any() and np.nanmin(after) < 0.95 * sm[k]:
        OC[s], OC_SRC[s] = OBINS[k] + 0.005, 1
    else:
        OC_SRC[s] = 2
fitted = OC_SRC == 1
OC_MED = float(np.median(OC[fitted]))
OC = np.where(OC_SRC == 2, OC_MED, OC)
OC = np.where(fitted, np.clip(OC, 0.05, 0.30), OC)
print(f'critical occupancy: fitted {int(fitted.sum())}, no congested branch → median {int((OC_SRC == 2).sum())}, '
      f'too little data {int((OC_SRC == 0).sum())}')
print(f'fitted o_c percentiles: p10 {np.percentile(OC[fitted], 10):.3f}  p50 {OC_MED:.3f}  p90 {np.percentile(OC[fitted], 90):.3f}')

critical occupancy: fitted 278, no congested branch → median 392, too little data 46
fitted o_c percentiles: p10 0.115  p50 0.155  p90 0.215


In [6]:
# ── L3 · Hourly labels → 15-min grid, class shares, face validity ─────────────
ratio = HOCC / OC[None, :]
LAB_H = np.select([ratio >= 1.0, ratio >= 0.7], [2, 1], 0).astype(np.int8)
LAB_H[~H_OK | ~np.isfinite(OC)[None, :]] = -1
LAB = np.repeat(LAB_H, 4, axis=0)                                  # (35040, N): each 15-min step takes its hour's class
CLASSES = ['free', 'heavy', 'congested']
save_factor('cong_label', label_hour=LAB_H, o_c=OC, o_c_src=OC_SRC)

spans = {'train': (0, TRAIN_END), 'val': (idx['val'][0] + 1, idx['val'][-1] + HORIZON + 1),
         'test': (idx['test'][0] + 1, T)}
rows = []
for k, (a, b) in spans.items():
    L = LAB[a:b]
    lab = L >= 0
    rows.append({'split': k, 'labelled': lab.mean(), **{c: (L == i).sum() / lab.sum() for i, c in enumerate(CLASSES)}})
shares = pd.DataFrame(rows).set_index('split')
print('class shares among labelled cells (ignored = unlabelled share):')
print(shares.round(4).to_string())

hour = TIMES.hour.to_numpy()
wk = np.asarray(TIMES.dayofweek < 5)
cong_by_hour = pd.DataFrame({'weekday': [(LAB[wk & (hour == h)] == 2).sum() / (LAB[wk & (hour == h)] >= 0).sum() for h in range(24)],
                             'weekend': [(LAB[~wk & (hour == h)] == 2).sum() / (LAB[~wk & (hour == h)] >= 0).sum() for h in range(24)]})
print('\ncongested share by hour of day (weekday | weekend):')
print('  ' + '  '.join(f'{h:02d}h {cong_by_hour.weekday[h]:.3f}|{cong_by_hour.weekend[h]:.3f}' for h in range(5, 21)))

smeta = pd.read_csv(PEMS / 'd11_station_meta_sd.csv').set_index('ID')
pk = wk & np.isin(hour, [6, 7, 8, 15, 16, 17, 18])
per_sensor = pd.Series((LAB[pk] == 2).sum(0) / np.maximum((LAB[pk] >= 0).sum(0), 1), index=meta['ID'])
top = per_sensor.sort_values(ascending=False).head(10)
print('\nmost congested sensors (share of weekday peak hours congested):')
for sid, v in top.items():
    r = smeta.loc[sid] if sid in smeta.index else None
    print(f'  {sid}  {meta.set_index("ID").loc[sid, "Fwy"]:>7}  {v:.2f}  {r["Name"] if r is not None else ""}')

day = TIMES.normalize()
tst = slice(*spans['test'])
hol = np.asarray(day[tst].isin(pd.date_range('2019-11-25', '2019-11-29'))) | np.asarray(day[tst] >= '2019-12-21')
wkt = wk[tst]
for name, m in (('regular test weekdays', wkt & ~hol), ('holiday-period weekdays', wkt & hol)):
    L = LAB[tst][m]
    print(f'{name:>24}: congested share {(L == 2).sum() / (L >= 0).sum():.4f}')
save_result('labels', {'shares': shares.round(5).to_dict(), 'o_c_median': OC_MED,
                       'o_c_sources': {'fitted': int(fitted.sum()), 'median': int((OC_SRC == 2).sum()), 'none': int((OC_SRC == 0).sum())}})

class shares among labelled cells (ignored = unlabelled share):
       labelled    free   heavy  congested
split                                     
train    0.8850  0.8633  0.0826     0.0541
val      0.8620  0.8546  0.0861     0.0594
test     0.8546  0.8727  0.0726     0.0547

congested share by hour of day (weekday | weekend):
  05h 0.005|0.000  06h 0.118|0.001  07h 0.211|0.001  08h 0.159|0.003  09h 0.053|0.009  10h 0.017|0.022  11h 0.018|0.037  12h 0.022|0.046  13h 0.029|0.043  14h 0.123|0.036  15h 0.264|0.033  16h 0.295|0.031  17h 0.276|0.026  18h 0.119|0.015  19h 0.010|0.005  20h 0.002|0.003

most congested sensors (share of weekday peak hours congested):
  1126022   SR78-W  0.78  78 WB E/O Nordahl
  1113279  SR163-S  0.65  ROBINSON AVE
  1111539  SR163-S  0.65  Robinson Ave
  1108680     I5-S  0.61  POINSETTIA LN
  1113308  SR125-N  0.61  125 NB CONNECTOR
  1108682     I5-N  0.60  POINSETTIA LN
  1108699   SR78-E  0.58  Twin Oaks Valley Rd
  1108704   SR78-W  0.57  Nordahl Rd
  

In [6]:
# ── L4 · What the hourly label loses: compare with 15-min occupancy labels on Nov 4–10 ──
wkq = (TIMES >= '2019-11-04') & (TIMES < '2019-11-11')
ok15 = (Q_N[wkq] > 0) & (Q_PO[wkq] >= 50) & np.isfinite(Q_OCC[wkq]) & np.isfinite(OC)[None, :]
r15 = Q_OCC[wkq] / OC[None, :]
L15 = np.select([r15 >= 1.0, r15 >= 0.7], [2, 1], 0)
Lh = LAB[wkq]
m = ok15 & (Lh >= 0)
cm = pd.crosstab(pd.Series(L15[m], name='15-min truth').map(dict(enumerate(CLASSES))),
                 pd.Series(Lh[m], name='hourly label').map(dict(enumerate(CLASSES))), normalize='index')
print(f'{m.sum():,} cells compared; overall agreement {np.mean(L15[m] == Lh[m]):.3f}')
print('rows = 15-min truth, columns = hourly label (row-normalised):')
print(cm.reindex(index=CLASSES, columns=CLASSES).round(3).to_string())
tp = ((L15 == 2) & (Lh == 2) & m).sum()
print(f'congested: recall of the hourly label {tp / ((L15 == 2) & m).sum():.3f}, precision {tp / ((Lh == 2) & m).sum():.3f}  '
      f'(15-min congested share {np.mean(L15[m] == 2):.4f} vs hourly {np.mean(Lh[m] == 2):.4f})')
save_result('labels_hourly_vs_15min', {'agreement': float(np.mean(L15[m] == Lh[m])), 'confusion': cm.round(4).to_dict()})

417,375 cells compared; overall agreement 0.957
rows = 15-min truth, columns = hourly label (row-normalised):
hourly label   free  heavy  congested
15-min truth                         
free          0.981  0.016      0.003
heavy         0.143  0.756      0.102
congested     0.007  0.093      0.900
congested: recall of the hourly label 0.900, precision 0.853  (15-min congested share 0.0678 vs hourly 0.0716)


In [7]:
# ── L5 · Validity: does congestion spread upstream like a queue? ──────────────
# Upstream neighbour = nearest sensor on the same freeway & direction within 3 postmiles on the upstream
# side (postmiles grow northbound/eastbound). Real queues grow backwards, so a sensor's congestion should
# raise its upstream neighbour's congestion, and more so one hour later than the reverse direction.
sm = smeta.reindex(meta['ID'])
pos = {sid: i for i, sid in enumerate(meta['ID'])}
pairs = []
for (fwy, d), g in sm.dropna(subset=['Abs_PM']).groupby(['Fwy', 'Dir']):
    g = g.sort_values('Abs_PM')
    ids_, pm = g.index.to_numpy(), g['Abs_PM'].to_numpy()
    for j, sid in enumerate(ids_):
        cand = range(j - 1, -1, -1) if d in ('N', 'E') else range(j + 1, len(ids_))
        for c in cand:
            gap = abs(pm[c] - pm[j])
            if 0 < gap <= 3:
                pairs.append((pos[sid], pos[ids_[c]]))                   # (downstream, upstream)
                break
            if gap > 3:
                break
down, up = np.array(pairs).T
Ld, Lu = LAB_H[:, down], LAB_H[:, up]
v = (Ld >= 0) & (Lu >= 0)
cd, cu = (Ld == 2) & v, (Lu == 2) & v
base = cu.sum() / v.sum()
p_same = (cd & cu).sum() / cd.sum()
v1 = v[:-1] & v[1:]
p_up_next = (cd[:-1] & cu[1:] & v1).sum() / (cd[:-1] & v1).sum()     # downstream now → upstream next hour
p_down_next = (cu[:-1] & cd[1:] & v1).sum() / (cu[:-1] & v1).sum()   # upstream now → downstream next hour
rng = np.random.default_rng(SEED)
fwy_of = meta['Fwy'].str[:-2].to_numpy()
far = np.array([rng.choice(np.nonzero(fwy_of != fwy_of[s])[0]) for s in down])    # same hours, different freeway
Lf = LAB_H[:, far]
vf = (Ld >= 0) & (Lf >= 0)
p_far = ((Ld == 2) & (Lf == 2) & vf).sum() / ((Ld == 2) & vf).sum()
print(f'{len(pairs)} downstream→upstream sensor pairs')
print(f'P(upstream congested) base rate {base:.3f}  |  given downstream congested, same hour {p_same:.3f} (lift ×{p_same / base:.1f})')
print(f'control, a sensor on a different freeway at the same hour: {p_far:.3f} (lift ×{p_far / base:.1f}) — the time-of-day effect alone')
print(f'one hour later: downstream→upstream {p_up_next:.3f} vs upstream→downstream {p_down_next:.3f}  '
      f'({"upstream spread dominates" if p_up_next > p_down_next else "no upstream asymmetry at hourly resolution"})')
save_result('labels_validity', {'pairs': len(pairs), 'base': base, 'p_same': p_same, 'p_far_control': p_far,
                                'p_up_next': p_up_next, 'p_down_next': p_down_next})

674 downstream→upstream sensor pairs
P(upstream congested) base rate 0.057  |  given downstream congested, same hour 0.731 (lift ×12.9)
control, a sensor on a different freeway at the same hour: 0.206 (lift ×3.6) — the time-of-day effect alone
one hour later: downstream→upstream 0.545 vs upstream→downstream 0.553  (no upstream asymmetry at hourly resolution)


In [8]:
# ── L6 · A flow-only label vs the occupancy label (the methodological finding) ──
# F2 from the proposal: flow below 70% of the sensor's own train-period Q85 for that slot and day type,
# for at least 30 min, only in slots where the road is normally busy (Q85 ≥ 0.5 × its train p99).
slot = (TIMES.hour * 4 + TIMES.minute // 15).to_numpy()
dtype3 = np.select([TIMES.dayofweek < 5, TIMES.dayofweek == 5], [0, 1], 2)
Ftr = np.where(F[:TRAIN_END] > 0, F[:TRAIN_END], np.nan)
CAP = np.nanpercentile(Ftr, 99, axis=0)
Q85 = np.full((3, STEPS_PER_DAY, N), np.nan, np.float32)
for d in range(3):
    for k in range(STEPS_PER_DAY):
        Q85[d, k] = np.nanpercentile(Ftr[(dtype3[:TRAIN_END] == d) & (slot[:TRAIN_END] == k)], 85, axis=0)
del Ftr
ref = Q85[dtype3, slot]                                             # (T, N)
low = (F > 0) & (ref >= 0.5 * CAP[None, :]) & (F < 0.7 * ref)
F2 = np.zeros_like(low)
F2[1:] = low[1:] & low[:-1]                                         # >= 2 consecutive 15-min steps
del ref, low

m = (LAB >= 0) & (F > 0)
occ_c = (LAB == 2) & m
tp = (F2 & occ_c).sum()
print(f'flow-only label vs occupancy truth (all labelled cells): precision {tp / (F2 & m).sum():.3f}, recall {tp / occ_c.sum():.3f}  '
      f'(flow-only flags {np.mean(F2[m]):.4f} of cells, occupancy congested {np.mean(occ_c[m]):.4f})')
for name, sel in (('regular test weekdays', wkt & ~hol), ('holiday-period weekdays', wkt & hol)):
    mm = m[tst][sel]
    print(f'{name:>24}: flow-only flags {np.mean(F2[tst][sel][mm]):.4f}   occupancy congested {np.mean((LAB[tst][sel] == 2)[mm]):.4f}')
xmas = np.asarray(day == '2019-12-25')
print(f'Christmas Day: flow-only flags {np.mean(F2[xmas][m[xmas]]):.4f} vs occupancy congested {np.mean((LAB[xmas] == 2)[m[xmas]]):.4f}')
save_result('labels_flow_only', {'precision': float(tp / (F2 & m).sum()), 'recall': float(tp / occ_c.sum())})

flow-only label vs occupancy truth (all labelled cells): precision 0.070, recall 0.031  (flow-only flags 0.0240 of cells, occupancy congested 0.0553)
   regular test weekdays: flow-only flags 0.0265   occupancy congested 0.0852
 holiday-period weekdays: flow-only flags 0.1763   occupancy congested 0.0293
Christmas Day: flow-only flags 0.4680 vs occupancy congested 0.0003


## ⛔ GATE L — STOP

Review L2–L6 before the label is used for training (R1, the congestion head):
- **o_c**: how many sensors were fitted vs given the median, and whether the fitted values look physical (loop detectors typically reach capacity somewhere around 10–20% occupancy).
- **Class shares**, split by train/val/test. They set the class weights, and show whether macro-F1 is needed (it almost certainly is).
- **Hourly vs 15-min**: how much congestion the hourly label misses or smears.
- **Validity**: lift over the different-freeway control, and upstream asymmetry.
- **Flow-only vs occupancy**, especially on holidays. That is the evidence that flow alone can't define congestion.

## Stage I · CHP incidents matched to sensors (CPU)

Run **00, 01, L1–L3** (for the labels used in the check), then **I1–I2**.

- **I1** does three things. It classifies each incident (collision / hazard / roadwork-or-traffic-break / other). It parses *lanes blocked* from the CHP dispatch notes. And it matches each incident to the sensors on the **same freeway and direction, from 3 mi upstream to 0.5 mi downstream**, active from its start until clearance.
- **Inputs use only what is known at the forecast origin:** whether an incident is active, minutes since it started, how far upstream it is, whether it's a collision, and lanes reported blocked *so far*. **Duration is never an input.** It only sets when an incident stops counting as active, and whether an incident has cleared is known in real time.
- **I2** checks validity with an event study. Does congestion at the matched sensors rise after an incident starts, relative to the same sensor at the same hour on other weekdays? It also reports how much of the test period is incident-affected, which becomes the incident / non-incident stratum.

In [6]:
# ── I1 · Classify, parse lanes blocked, match to sensors, build incident features ──
import re
YEAR0 = TIMES[0]
inc = pd.read_csv(PEMS / 'chp_incidents_d11_2019.csv')
det = pd.read_csv(PEMS / 'chp_incident_details_d11_2019.csv')
inc['ts'] = pd.to_datetime(inc['timestamp'], format='%m/%d/%Y %H:%M:%S')
inc['dur'] = inc['duration'].fillna(inc['duration'].median()).clip(lower=0)
d = inc['description'].fillna('')
inc['kind'] = np.select([d.str.contains(r'Collision|Hit and Run|SPINOUT|^118[0-3]|^1179', case=False),
                         d.str.contains(r'Hazard|Animal|Debris', case=False),
                         d.str.contains(r'CZP|MZP|BREAK|Construction|Maintenance', case=False)],
                        ['collision', 'hazard', 'work/break'], 'other')

# lanes blocked: a blocking word AND a lane reference ("BLK" alone is usually a car colour)
blocking = det['description'].str.contains(r'BLKG|BLKING|BLKD|BLOCKING|BLOCKED|BLOCKS|\bBLK\s*#|CLOSED', case=False, na=False) & \
           ~det['description'].str.contains(r'\bOPEN', case=False, na=False)
lanes_txt = det.loc[blocking, 'description']
nums = lanes_txt.str.findall(r'#\s*([1-8])').apply(lambda v: len(set(v)))
alls = lanes_txt.str.contains(r'ALL\s*(?:LNS|LANES)', case=False, na=False)
bl = pd.DataFrame({'incident_id': det.loc[blocking, 'incident_id'],
                   'ts': pd.to_datetime(det.loc[blocking, 'timestamp'], format='%m/%d/%Y %H:%M:%S'),
                   'n': np.where(alls, 9, nums)})
bl = bl[bl['n'] > 0].sort_values('ts').groupby('incident_id').first()      # first report of blocked lanes
inc = inc.join(bl.rename(columns={'ts': 'blk_ts', 'n': 'blk_n'}), on='incident_id')
print(f"{len(inc):,} D11 incidents; kinds {inc['kind'].value_counts().to_dict()}")
print(f"lanes-blocked reported for {inc['blk_n'].notna().mean():.3f} of incidents "
      f"({inc.loc[inc['kind'] == 'collision', 'blk_n'].notna().mean():.3f} of collisions); 'all lanes' in {(inc['blk_n'] == 9).sum()}")

# match: same freeway + direction, sensor between 3 mi upstream and 0.5 mi downstream of the incident
sm = smeta.reindex(meta['ID'])
spm, sfwy, sdir = sm['Abs_PM'].to_numpy(), sm['Fwy'].to_numpy(), sm['Dir'].to_numpy()
step = pd.Timedelta(minutes=15)
INC_ACTIVE = np.zeros((T, N), np.uint8)
INC_COLL = np.zeros((T, N), bool)
INC_SINCE = np.zeros((T, N), np.float32)                 # minutes since start of the most recent active incident
INC_UP = np.full((T, N), np.inf, np.float32)             # miles the incident lies downstream of the sensor
INC_BLK = np.zeros((T, N), np.uint8)                     # lanes reported blocked so far (9 = all)
matched = 0
for r in inc.itertuples():
    same = (sfwy == r.fwy) & (sdir == r.dir)
    up = (r.abs_pm - spm) if r.dir in ('N', 'E') else (spm - r.abs_pm)   # > 0: sensor is upstream
    hit = np.nonzero(same & (up >= -0.5) & (up <= 3.0))[0]
    if not len(hit):
        continue
    matched += 1
    s0 = int((r.ts - YEAR0) // step)
    s1 = int((r.ts + pd.Timedelta(minutes=r.dur) - YEAR0) // step)
    if s0 >= T or s1 < 0:
        continue
    ks = np.arange(max(s0, 0), min(s1, T - 1) + 1)
    since = ((ks + 1) * 15 - (r.ts - YEAR0).total_seconds() / 60)[:, None]  # at the end of each window
    INC_ACTIVE[np.ix_(ks, hit)] += 1
    if r.kind == 'collision':
        INC_COLL[np.ix_(ks, hit)] = True
    cur = INC_SINCE[np.ix_(ks, hit)]
    INC_SINCE[np.ix_(ks, hit)] = np.where(cur > 0, np.minimum(cur, since), since)
    INC_UP[np.ix_(ks, hit)] = np.minimum(INC_UP[np.ix_(ks, hit)], np.abs(up[hit])[None, :])
    if not np.isnan(r.blk_n):
        kb = ks[(ks + 1) * 15 > (r.blk_ts - YEAR0).total_seconds() / 60]   # only once the blocking has been reported
        if len(kb):
            INC_BLK[np.ix_(kb, hit)] = np.maximum(INC_BLK[np.ix_(kb, hit)], int(r.blk_n))
INC_UP[~np.isfinite(INC_UP)] = 0
print(f'incidents matched to ≥1 sensor: {matched:,} ({matched / len(inc):.3f}); sensors without postmiles: '
      f'{int(np.isnan(spm).sum())}')
aff = INC_ACTIVE > 0
spans_c = {k: slice(a, b) for k, (a, b) in spans.items()}
print('incident-affected sensor-steps: ' + '   '.join(f'{k} {aff[s].mean():.4f}' for k, s in spans_c.items()))
save_factor('incidents', active=INC_ACTIVE, collision=INC_COLL, since=INC_SINCE.astype(np.float16),
            up_miles=INC_UP.astype(np.float16), lanes_blocked=INC_BLK)

49,198 D11 incidents; kinds {'hazard': 25658, 'collision': 17927, 'other': 3474, 'work/break': 2139}
lanes-blocked reported for 0.097 of incidents (0.155 of collisions); 'all lanes' in 183


incidents matched to ≥1 sensor: 45,573 (0.926); sensors without postmiles: 8
incident-affected sensor-steps: train 0.0282   val 0.0278   test 0.0307


PosixPath('/Users/lavanyapareek/Documents/Capstone_Post_Joining/work/factors/incidents.npz')

In [7]:
# ── I2 · Validity: does congestion rise after incidents start at the matched sensors? ──
# Event study on weekday daytime collisions: congested share of the nearest upstream sensor (0–1.5 mi)
# at hours -2..+3 around the start, minus the same sensor's congested share at the same hour of day on
# other weekdays (so rush-hour timing alone cannot produce a rise).
wkday = np.asarray(TH.dayofweek < 5)
hod = np.asarray(TH.hour)
base = np.full((24, N), np.nan)
for h_ in range(24):
    L = LAB_H[wkday & (hod == h_)]
    base[h_] = np.where((L >= 0).sum(0) > 0, (L == 2).sum(0) / np.maximum((L >= 0).sum(0), 1), np.nan)
rel = np.arange(-2, 4)
diffs = {k: [] for k in rel}
for r in inc[(inc['kind'] == 'collision') & (inc['ts'].dt.dayofweek < 5) & inc['ts'].dt.hour.between(6, 18)].itertuples():
    same = (sfwy == r.fwy) & (sdir == r.dir)
    up = (r.abs_pm - spm) if r.dir in ('N', 'E') else (spm - r.abs_pm)
    cand = np.nonzero(same & (up >= 0) & (up <= 1.5))[0]
    if not len(cand):
        continue
    s = cand[np.argmin(up[cand])]
    h0 = int((r.ts - YEAR0) // pd.Timedelta(hours=1))
    for k in rel:
        h = h0 + k
        if 0 <= h < 8760 and LAB_H[h, s] >= 0 and np.isfinite(base[hod[h], s]):
            diffs[k].append(float(LAB_H[h, s] == 2) - base[hod[h], s])
print('excess congested share at the nearest upstream sensor vs its usual level at that hour (weekday daytime collisions):')
print('  ' + '   '.join(f'{k:+d}h {np.mean(diffs[k]):+.3f} (n={len(diffs[k]):,})' for k in rel))
inc_cells = INC_ACTIVE[spans_c['test']] > 0
L = LAB[spans_c['test']]
print(f'test period: congested share in incident-affected cells {(L[inc_cells] == 2).sum() / (L[inc_cells] >= 0).sum():.3f} '
      f'vs {(L[~inc_cells] == 2).sum() / (L[~inc_cells] >= 0).sum():.3f} elsewhere')
save_result('incidents', {'n': int(len(inc)), 'matched': int(matched), 'kinds': inc['kind'].value_counts().to_dict(),
                          'blk_coverage': float(inc['blk_n'].notna().mean()),
                          'event_study': {int(k): float(np.mean(v)) for k, v in diffs.items()},
                          'affected_share': {k: float(aff[s].mean()) for k, s in spans_c.items()}})

excess congested share at the nearest upstream sensor vs its usual level at that hour (weekday daytime collisions):
  -2h +0.026 (n=8,730)   -1h +0.061 (n=8,722)   +0h +0.142 (n=8,722)   +1h +0.098 (n=8,732)   +2h +0.039 (n=8,744)   +3h +0.017 (n=8,737)
test period: congested share in incident-affected cells 0.148 vs 0.052 elsewhere


### Incidents — review before feature 4d
The I1 and I2 outputs: match rate, lanes-blocked coverage, and the event study. The key question is whether congestion at the matched sensor rises clearly above its usual level for that hour once a collision has started. If so, both the spatial matching and the time alignment are right, and the incident features and stratum can be used.

## Stage F · Feature assembly for the ablation rows R2–R6 (CPU)

Run **00, 01**, then **F1–F4** (each writes a small factor file), then **F5**, which writes `features.py`, and **F6**, which tests every level. Levels are cumulative. Each row adds one change, and all of them keep the congestion head:

| Row | Adds | Inputs |
|---|---|---|
| R2 | data fixes | flow with missing inputs filled causally from the same slot on the previous 7 days, plus a missing-data mask |
| R3 | 4a | utilisation (flow ÷ train p99), flow per lane, sin/cos time of day, one-hot day of week, static lanes / freeway / direction / location |
| R4 | 4b | US + CA holidays (major ones treated as a Sunday), day before / after, holiday period, SDUSD school in session |
| R5 | 4c | weather from 5 ASOS stations: rain now, hourly and 3/6/24 h accumulation, hours since rain, fog, low visibility; causal carry-forward, inverse-distance weighted to each sensor |
| R6 | 4d | incident active nearby, collision, minutes since start, miles upstream of the incident |

**The inputs stay factored**, per sensor-time, per time, per sensor and per weather station, and are joined per batch. Fully expanded, R6 would need about 2 GB on the 8 GB M2. **The flow target is never taken from these inputs.**

In [7]:
# ── F1 · R2: missing-data mask + causal fill for missing input flow (fill chosen on hidden val cells) ──
def runs(mask):
    d = np.diff(np.pad(mask.T.astype(np.int8), ((0, 0), (1, 1))), axis=1)
    s_sensor, s_t = np.nonzero(d == 1)
    _, e_t = np.nonzero(d == -1)
    return s_sensor, s_t, e_t - s_t

Z = F == 0
same = np.zeros_like(Z)
same[1:] = (F[1:] == F[:-1]) & (F[1:] > 0) & (F[1:] != 999)
ss, st, sl = runs(same)
STUCK2 = np.zeros_like(Z)
for s, t0, L in zip(ss[sl >= 7], st[sl >= 7], sl[sl >= 7]):          # >= 8 identical values = 2 h
    STUCK2[t0 - 1:t0 + L, s] = True
MISS = Z | STUCK2

slot = (TIMES.hour * 4 + TIMES.minute // 15).to_numpy()
dtype3 = np.select([TIMES.dayofweek < 5, TIMES.dayofweek == 5], [0, 1], 2)
PROFILE = load_factor('profile_median_train')['profile']              # train-only (cell 05)

def lag_median(Fv, t0, t1, lags_days):
    """Median over the same 15-min slot on the given past days — strictly past values only."""
    lags = np.full((len(lags_days), t1 - t0, Fv.shape[1]), np.nan, np.float32)
    for j, k in enumerate(lags_days):
        src = np.arange(t0, t1) - 96 * k
        ok = src >= 0
        lags[j, ok] = Fv[src[ok]]
    return np.nanmedian(lags, axis=0)

Fv = np.where(MISS, np.nan, F).astype(np.float32)
PROF_T = PROFILE[dtype3, slot]
CANDIDATES = {'previous 7 days': range(1, 8), 'same weekday, last 4 weeks': (7, 14, 21, 28)}
fills = {}
for name, lags_days in CANDIDATES.items():
    arr = np.full((T, N), np.nan, np.float32)
    for c0 in range(0, T, 4096):
        arr[c0:c0 + 4096] = lag_median(Fv, c0, min(c0 + 4096, T), lags_days)
    fills[name] = np.where(np.isnan(arr), PROF_T, arr)                 # train profile where the past is missing too
fills['train profile only'] = PROF_T

# choose the fill on hidden validation cells (1% of observed cells, reconstructed from each candidate)
rng = np.random.default_rng(SEED)
va = slice(idx['val'][0], idx['val'][-1])
hide = (rng.random(F[va].shape) < 0.01) & ~MISS[va]
scores = {k: float(np.abs(v[va][hide] - F[va][hide]).mean()) for k, v in fills.items()}
FILL_NAME = min(scores, key=scores.get)
print(f'fill check on {hide.sum():,} hidden val cells (MAE): ' + '   '.join(f'{k} {v:.1f}' for k, v in scores.items())
      + f'   zero, as LargeST does {float(np.abs(F[va][hide]).mean()):.1f}   → using "{FILL_NAME}"')
FLOW_IN = np.where(MISS, fills[FILL_NAME], F).astype(np.float32)
del fills
assert np.array_equal(FLOW_IN[~MISS], F[~MISS]), 'observed inputs must be untouched'
print(f'missing inputs: zeros {Z.mean():.4f} + stuck ≥2 h {(STUCK2 & ~Z).mean():.4f} = {MISS.mean():.4f} of cells')
save_factor('inputs_r2', flow_in=FLOW_IN, miss=MISS, fill_method=np.array(FILL_NAME))

fill check on 49,053 hidden val cells (MAE): previous 7 days 34.6   same weekday, last 4 weeks 16.9   train profile only 21.9   zero, as LargeST does 255.0   → using "same weekday, last 4 weeks"
missing inputs: zeros 0.0225 + stuck ≥2 h 0.0003 = 0.0228 of cells


In [4]:
# ── F2 · 4a: utilisation, flow per lane, cyclical time, static sensor attributes ──
Ftr = np.where(MISS[:TRAIN_END], np.nan, F[:TRAIN_END])
CAP = np.nanpercentile(Ftr, 99, axis=0)
CAP = np.where(np.isfinite(CAP) & (CAP > 0), CAP, np.nanmedian(CAP)).astype(np.float32)   # dead sensors
del Ftr
EFF_LANES = LANES.astype(np.float32).copy()
EFF_LANES[meta['ID'].to_numpy() == 1108333] = 5                     # Gate 0 decision: listed 2 lanes, carries 5
UTIL = FLOW_IN / CAP[None, :]
FPL = FLOW_IN / EFF_LANES[None, :] / 200.0                           # ~per-lane capacity per 5 min
tod = slot / STEPS_PER_DAY
TIME_4A = np.column_stack([np.sin(2 * np.pi * tod), np.cos(2 * np.pi * tod),
                           np.eye(7)[TIMES.dayofweek.to_numpy()]]).astype(np.float32)
fwy = meta['Fwy'].str.rsplit('-', n=1).str[0]
fwy_levels, dir_levels = sorted(fwy.unique()), ['N', 'S', 'E', 'W']
lat, lng = meta['Lat'].to_numpy(), meta['Lng'].to_numpy()
STATIC = np.column_stack([EFF_LANES / 8, np.eye(len(fwy_levels))[fwy.map({f: i for i, f in enumerate(fwy_levels)})],
                          np.eye(4)[meta['Direction'].map({d_: i for i, d_ in enumerate(dir_levels)})],
                          (lat - lat.mean()) / lat.std(), (lng - lng.mean()) / lng.std()]).astype(np.float32)
STATIC_NAMES = ['lanes'] + [f'fwy_{f}' for f in fwy_levels] + [f'dir_{d_}' for d_ in dir_levels] + ['lat', 'lng']
print(f'utilisation p50 {np.percentile(UTIL, 50):.2f} p99 {np.percentile(UTIL, 99):.2f} | flow/lane p99 {np.percentile(FPL, 99):.2f} | '
      f'static channels {STATIC.shape[1]}: {STATIC_NAMES}')
save_factor('inputs_4a', util=UTIL.astype(np.float16), fpl=FPL.astype(np.float16), time=TIME_4A, static=STATIC,
            static_names=np.array(STATIC_NAMES), cap=CAP, eff_lanes=EFF_LANES)

utilisation p50 0.49 p99 1.00 | flow/lane p99 0.76 | static channels 20: ['lanes', 'fwy_I15', 'fwy_I5', 'fwy_I8', 'fwy_I805', 'fwy_I905', 'fwy_SR11', 'fwy_SR125', 'fwy_SR163', 'fwy_SR52', 'fwy_SR54', 'fwy_SR56', 'fwy_SR78', 'fwy_SR94', 'dir_N', 'dir_S', 'dir_E', 'dir_W', 'lat', 'lng']


PosixPath('/Users/lavanyapareek/Documents/Capstone_Post_Joining/work/factors/inputs_4a.npz')

In [5]:
# ── F3 · 4b: holidays, day before/after, holiday period, SDUSD school in session ──
import holidays as _hol
HOL = {pd.Timestamp(d): n for d, n in _hol.US(state='CA', years=[2018, 2019, 2020]).items()}
MAJOR = {"New Year's Day", 'Memorial Day', 'Independence Day', 'Labor Day', 'Thanksgiving Day', 'Christmas Day'}
days = pd.date_range('2019-01-01', '2019-12-31')
is_hol = days.isin(list(HOL))
before = (days + pd.Timedelta(days=1)).isin(list(HOL))
after = (days - pd.Timedelta(days=1)).isin(list(HOL))
major = np.array([HOL.get(d_, '') in MAJOR for d_ in days])
period = ((days.month == 12) & (days.day >= 23)) | ((days.month == 1) & (days.day <= 4))   # winter break, both years
off = pd.DatetimeIndex(np.concatenate([
    pd.date_range('2019-01-01', '2019-01-04'), pd.to_datetime(['2019-01-21', '2019-02-15', '2019-02-18']),
    pd.date_range('2019-03-25', '2019-03-29'), pd.to_datetime(['2019-05-24', '2019-05-27']),
    pd.date_range('2019-06-12', '2019-08-25'),                         # summer: after Jun 11, before Aug 26
    pd.to_datetime(['2019-09-02', '2019-11-11']), pd.date_range('2019-11-25', '2019-11-29'),
    pd.date_range('2019-12-23', '2019-12-31')]))
school = (days.dayofweek < 5) & ~days.isin(off)
eff_dow = np.where(major, 6, days.dayofweek)                          # major holiday -> behaves like a Sunday
per_day = np.column_stack([is_hol, before, after, period, school]).astype(np.float32)
di = (TIMES.normalize() - TIMES[0]).days.to_numpy()
TIME_4B = per_day[di]
DOW_REMAP = np.eye(7, dtype=np.float32)[eff_dow[di]]
print('holidays:', ', '.join(f'{d_:%b %d} {HOL[d_]}{" *" if HOL[d_] in MAJOR else ""}' for d_ in days[is_hol]), '(* = treated as Sunday)')
print(f'school-in-session weekdays: {int(school.sum())}   holiday-period days: {int(period.sum())}')
save_factor('inputs_4b', time=TIME_4B, dow_remap=DOW_REMAP,
            names=np.array(['holiday', 'day_before', 'day_after', 'holiday_period', 'school']))

holidays: Jan 01 New Year's Day *, Jan 21 Martin Luther King Jr. Day, Feb 15 Susan B. Anthony Day, Feb 18 Presidents' Day, Mar 31 Cesar Chavez Day, Apr 01 Cesar Chavez Day (observed), May 27 Memorial Day *, Jul 04 Independence Day *, Sep 02 Labor Day *, Nov 11 Veterans Day, Nov 28 Thanksgiving Day *, Nov 29 Day After Thanksgiving, Dec 25 Christmas Day * (* = treated as Sunday)
school-in-session weekdays: 180   holiday-period days: 13


PosixPath('/Users/lavanyapareek/Documents/Capstone_Post_Joining/work/factors/inputs_4b.npz')

In [6]:
# ── F4 · 4c: weather from 5 San Diego ASOS stations (Iowa State archive), causal, IDW ──
import io, urllib.request
WX_DIR = DRIVE_PROJ / 'weather'
WX_DIR.mkdir(exist_ok=True)
STATIONS = ['SAN', 'MYF', 'SEE', 'CRQ', 'NKX']
raw_csv = WX_DIR / 'asos_2019_routine.csv'
if not raw_csv.exists():                                            # one request; routine hourly METARs only
    q = '&'.join([f'station={s_}' for s_ in STATIONS] + [f'data={v}' for v in ('tmpf', 'p01i', 'vsby', 'wxcodes')] +
                 ['year1=2018', 'month1=12', 'day1=29', 'year2=2020', 'month2=1', 'day2=2', 'tz=Etc/UTC',
                  'format=onlycomma', 'latlon=yes', 'missing=empty', 'trace=0.0001', 'report_type=3'])
    raw_csv.write_bytes(urllib.request.urlopen(f'https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?{q}', timeout=300).read())
wx = pd.read_csv(raw_csv)
wx['utc'] = pd.to_datetime(wx['valid']).dt.tz_localize('UTC')
wx['local'] = wx['utc'].dt.tz_convert('America/Los_Angeles')
assert wx.loc[wx['utc'].dt.month == 1, 'local'].iloc[0].utcoffset() == pd.Timedelta(hours=-8)
assert wx.loc[wx['utc'].dt.month == 7, 'local'].iloc[0].utcoffset() == pd.Timedelta(hours=-7)
wx['naive'] = wx['local'].dt.tz_localize(None)                     # traffic index is naive local wall-clock (Gate 0)
for m_ in (1, 7):                                                  # daily temperature peak must land in the afternoon
    g = wx[(wx['naive'].dt.month == m_) & wx['tmpf'].notna()]
    pk = g.loc[g.groupby([g['station'], g['naive'].dt.date])['tmpf'].idxmax(), 'naive'].dt.hour.median()
    assert 11 <= pk <= 17, f'month {m_}: temperature peaks at {pk} h local — timezone handling is wrong'
    print(f'month {m_}: median hour of daily max temperature {pk:.0f}:00 local ✓')

codes = wx['wxcodes'].fillna('')
wx['rain'] = (wx['p01i'].fillna(0) > 0) | codes.str.contains(r'RA|DZ|SH|TS|UP')
wx['fog'] = codes.str.contains(r'FG')
wx['lowvis'] = wx['vsby'] < 1
wx['p01i'] = wx['p01i'].fillna(0)
win_end = TIMES + pd.Timedelta(minutes=15)                         # a window's value may use obs up to its end
feat = []
for s_ in STATIONS:
    g = wx[wx['station'] == s_].sort_values('naive').set_index('naive')
    acc = {h: g['p01i'].rolling(f'{h}h').sum() for h in (3, 6, 24)}
    last_rain = g.index.to_series().where(g['rain']).ffill()
    hs = (g.index.to_series() - last_rain).dt.total_seconds() / 3600
    hourly = pd.DataFrame({'rain': g['rain'].astype(float), 'p01i': g['p01i'], 'acc3': acc[3], 'acc6': acc[6], 'acc24': acc[24],
                           'since': np.log1p(hs.fillna(72).clip(upper=72)) / np.log1p(72), 'fog': g['fog'].astype(float),
                           'lowvis': g['lowvis'].astype(float)}).reset_index()
    al = pd.merge_asof(pd.DataFrame({'naive': win_end}), hourly.sort_values('naive'), on='naive', direction='backward',
                       tolerance=pd.Timedelta(hours=2))
    feat.append(al.drop(columns='naive').to_numpy(np.float32))
WX = np.stack(feat, axis=1)                                        # (T, S, k)
WX_NAMES = ['rain', 'p01i', 'acc3', 'acc6', 'acc24', 'since', 'fog', 'lowvis']
miss_st = np.isnan(WX)
WX = np.where(miss_st, np.nanmean(WX, axis=1, keepdims=True), WX)  # a silent station borrows the network mean
WX = np.nan_to_num(WX, nan=0.0)
WX[..., 1:5] *= 10                                                 # inches -> tenths, O(1) scale

lat, lng = meta['Lat'].to_numpy(), meta['Lng'].to_numpy()
st_ll = wx.groupby('station')[['lat', 'lon']].first().loc[STATIONS].to_numpy()
la1, lo1 = np.radians(lat)[:, None], np.radians(lng)[:, None]
la2, lo2 = np.radians(st_ll[:, 0])[None, :], np.radians(st_ll[:, 1])[None, :]
dist = 6371 * 2 * np.arcsin(np.sqrt(np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2))
W_IDW = (1 / np.maximum(dist, 1) ** 2)
W_IDW = (W_IDW / W_IDW.sum(1, keepdims=True)).astype(np.float32)   # (N, S)
rain_net = WX[..., 0] @ W_IDW.mean(0)
print(f'station obs missing: {miss_st[..., 0].mean(0).round(3).tolist()} (share of 15-min steps, per station)')
tr_end, te_start = TRAIN_END, idx['test'][0] + 1
print(f'rainy 15-min steps (network IDW rain > 0.5): train {np.mean(rain_net[:tr_end] > 0.5):.3f}  '
      f'val {np.mean(rain_net[tr_end:te_start] > 0.5):.3f}  test {np.mean(rain_net[te_start:] > 0.5):.3f}  '
      f'| test rain hours ≈ {np.sum(rain_net[te_start:] > 0.5) / 4:.0f}')
save_factor('inputs_4c', wx=WX.astype(np.float16), w_idw=W_IDW, names=np.array(WX_NAMES), stations=np.array(STATIONS))

month 1: median hour of daily max temperature 12:00 local ✓
month 7: median hour of daily max temperature 12:00 local ✓
station obs missing: [0.0, 0.0, 0.027, 0.002, 0.02] (share of 15-min steps, per station)
rainy 15-min steps (network IDW rain > 0.5): train 0.073  val 0.006  test 0.084  | test rain hours ≈ 147


PosixPath('/Users/lavanyapareek/Documents/Capstone_Post_Joining/work/factors/inputs_4c.npz')

In [5]:
%%writefile {WORK}/cong/features.py
# ── F5 · features.py — assemble factored inputs for each ablation level ─────
"""Factored model inputs per ablation level, built from the Stage F factor files. Levels are cumulative:
R2 data fixes, R3 +4a, R4 +4b, R5 +4c weather, R6 +4d incidents. Channel 0 of `dyn` is always the
normalised flow (LargeST's mean/std); the flow *target* is never taken from these inputs."""
import numpy as np

LEVELS = ['R0', 'R2', 'R3', 'R4', 'R5', 'R6']


def assemble(level, data_dir, fact_dir):
    """-> dict(dyn (T,N,Cd) float16, time (T,Ct) | None, static (N,Cs) | None, wx (T,S,k) | None,
    wx_w (N,S) | None, miss (T,N) bool, names [dyn..., time..., static..., wx...])"""
    lv = LEVELS.index(level)
    assert lv >= 1, 'R0 uses his.npz directly'
    with np.load(data_dir / '2019' / 'his.npz') as z:
        mean, std = float(z['mean']), float(z['std'])
        largest_time = z['data'][:, 0, 1:3].astype(np.float32)          # LargeST's (tod, dow/7), same for every sensor
    r2 = np.load(fact_dir / 'inputs_r2.npz')
    dyn, names = [(r2['flow_in'] - mean) / std, r2['miss'].astype(np.float32)], ['flow', 'missing']
    time, t_names, static, s_names, wx, wx_w, w_names = largest_time, ['tod', 'dow/7'], None, [], None, None, []
    if lv >= 2:                                                          # R3: 4a
        a = np.load(fact_dir / 'inputs_4a.npz')
        dyn += [a['util'].astype(np.float32), a['fpl'].astype(np.float32)]
        names += ['util', 'flow_per_lane']
        time, t_names = a['time'], ['tod_sin', 'tod_cos'] + [f'dow_{i}' for i in range(7)]
        static, s_names = a['static'], [str(x) for x in a['static_names']]
    if lv >= 3:                                                          # R4: 4b (major holidays -> Sunday in the one-hot)
        b = np.load(fact_dir / 'inputs_4b.npz')
        time = np.concatenate([time[:, :2], b['dow_remap'], b['time']], axis=1)
        t_names = t_names[:2] + [f'dow_{i}' for i in range(7)] + [str(x) for x in b['names']]
    if lv >= 4:                                                          # R5: 4c weather
        c = np.load(fact_dir / 'inputs_4c.npz')
        wx, wx_w, w_names = c['wx'], c['w_idw'], [f'wx_{x}' for x in c['names']]
    if lv >= 5:                                                          # R6: 4d incidents (known at the forecast origin)
        i = np.load(fact_dir / 'incidents.npz')
        dyn += [np.minimum(i['active'], 3) / 3, i['collision'].astype(np.float32),
                np.log1p(i['since'].astype(np.float32)) / np.log1p(600), i['up_miles'].astype(np.float32) / 3]
        names += ['inc_active', 'inc_collision', 'inc_since', 'inc_up_miles']
    return {'dyn': np.stack(dyn, axis=-1).astype(np.float16), 'time': time, 'static': static, 'wx': wx, 'wx_w': wx_w,
            'miss': r2['miss'], 'names': names + t_names + s_names + w_names}

Overwriting /Users/lavanyapareek/Documents/Capstone_Post_Joining/work/cong/features.py


In [6]:
# ── F6 · Test every level: guard, batch shape, one real training step on the device ──
import importlib
sys.path.insert(0, str(WORK / 'cong'))
import features
importlib.reload(features)
importlib.reload(conglib)
DEVICE = conglib.pick_device()
MICRO_BS = 8 if DEVICE.type == 'mps' else None
LABF = np.repeat(load_factor('cong_label')['label_hour'], 4, axis=0)
cw = torch.tensor([0.365, 1.179, 1.457], device=DEVICE)
rows = []
for lvl in features.LEVELS[1:]:
    fin = features.assemble(lvl, DATA, FACT)
    ok = ~fin['miss']
    err = np.abs(fin['dyn'][..., 0][ok].astype(np.float32) * STD + MEAN - F[ok]).max()
    assert err < 0.6, f'{lvl}: flow channel differs from the target by {err}'
    d = conglib.Windows(fin['dyn'], F, DEVICE, x_dtype=torch.float16, Yc=LABF, time=fin['time'], static=fin['static'],
                        wx=fin['wx'], wx_w=fin['wx_w'])
    x, y, yc = d.batch(idx['train'][:64])
    assert x.shape[-1] == d.n_channels == len(fin['names']), (x.shape, d.n_channels, len(fin['names']))
    assert torch.equal(y, d.Y[torch.as_tensor(idx['train'][:64], device=DEVICE)[:, None] + d.y_off]), 'targets must come from F'
    conglib.set_seed(SEED)
    m = conglib.GWNetMT(build_gwnet(adj, 'doubletransition', 1, d.n_channels, DEVICE, REPO))
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    sc = torch.amp.GradScaler(DEVICE.type, enabled=False)
    conglib.reset_mem(DEVICE)
    conglib.train_step_mt(m, opt, sc, x, y, yc, MEAN, STD, False, 15.0, cw, micro_bs=MICRO_BS)   # warm-up
    conglib.sync(DEVICE)
    t0 = time.time()
    mae, ce = conglib.train_step_mt(m, opt, sc, *d.batch(idx['train'][64:128]), MEAN, STD, False, 15.0, cw, micro_bs=MICRO_BS)
    conglib.sync(DEVICE)
    rows.append({'level': lvl, 'channels': d.n_channels, 'dyn MiB': round(d.X.element_size() * d.X.nelement() / 2**20),
                 'ms/step': round(1000 * (time.time() - t0)), 'mem GiB': round(conglib.mem_gib(DEVICE), 2),
                 'params': sum(p.numel() for p in m.parameters()), 'flow guard err': round(float(err), 3)})
    print(f"{lvl}: {d.n_channels} channels — {', '.join(fin['names'])}")
    del d, m, opt, fin, x, y, yc
    conglib.reset_mem(DEVICE)
print()
print(pd.DataFrame(rows).to_string(index=False))

check supports length 2 3


R2: 4 channels — flow, missing, tod, dow/7


check supports length 2 3


R3: 33 channels — flow, missing, util, flow_per_lane, tod_sin, tod_cos, dow_0, dow_1, dow_2, dow_3, dow_4, dow_5, dow_6, lanes, fwy_I15, fwy_I5, fwy_I8, fwy_I805, fwy_I905, fwy_SR11, fwy_SR125, fwy_SR163, fwy_SR52, fwy_SR54, fwy_SR56, fwy_SR78, fwy_SR94, dir_N, dir_S, dir_E, dir_W, lat, lng


check supports length 2 3


...

check supports length 2 3


R5: 46 channels — flow, missing, util, flow_per_lane, tod_sin, tod_cos, dow_0, dow_1, dow_2, dow_3, dow_4, dow_5, dow_6, holiday, day_before, day_after, holiday_period, school, lanes, fwy_I15, fwy_I5, fwy_I8, fwy_I805, fwy_I905, fwy_SR11, fwy_SR125, fwy_SR163, fwy_SR52, fwy_SR54, fwy_SR56, fwy_SR78, fwy_SR94, dir_N, dir_S, dir_E, dir_W, lat, lng, wx_rain, wx_p01i, wx_acc3, wx_acc6, wx_acc24, wx_since, wx_fog, wx_lowvis


check supports length 2 3


R6: 50 channels — flow, missing, util, flow_per_lane, inc_active, inc_collision, inc_since, inc_up_miles, tod_sin, tod_cos, dow_0, dow_1, dow_2, dow_3, dow_4, dow_5, dow_6, holiday, day_before, day_after, holiday_period, school, lanes, fwy_I15, fwy_I5, fwy_I8, fwy_I805, fwy_I905, fwy_SR11, fwy_SR125, fwy_SR163, fwy_SR52, fwy_SR54, fwy_SR56, fwy_SR78, fwy_SR94, dir_N, dir_S, dir_E, dir_W, lat, lng, wx_rain, wx_p01i, wx_acc3, wx_acc6, wx_acc24, wx_since, wx_fog, wx_lowvis



level  channels  dyn MiB  ms/step  mem GiB  params  flow guard err
   R2         4       96     2999     3.09  329664           0.356
   R3        33      191     2977     3.07  330592           0.356
   R4        38      191     2985     3.07  330752           0.356
   R5        46      191     3053     3.07  331008           0.356
   R6        50      383     3001     3.07  331136           0.356


## Stage E · Final evaluation (classification baselines, stratified error, bootstrap CIs)

Run **00, 01, 07, 08, F1–F6** first (needed globals: `F`, `adj`, `TIMES`, `dtype3`, `slot`, `TRAIN_END`, `LAB`, `DEVICE`, `EVAL_BS`), then **E2–E7**. `conglib.py` (cell 07) gained two functions for this stage — `load_run` (rebuilds any finished run's exact model + inputs from its own `ckpt/<run>/best.pt`, since every checkpoint stores its own training args) and `predict_mt` (one forward pass over the test set, caching flow + class predictions so every stratified/bootstrap number below comes from numpy on that cache instead of a separate pass per question). Nothing here retrains or writes to `ckpt/*` — read + inference only.

- **E2** computes the two classification baselines the brief calls for (persistence, profile-class) so R1–R6's saved macro-F1 has something to compare against.
- **E3** builds the rain / incident / peak / holiday-period / true-class masks from the existing factor files.
- **E4** is the one real compute step: one forward pass per of the 10 finished runs, then the full stratified flow-MAE table and the day-block bootstrap primitive, both from that single cached pass.
- **E5** turns E4's per-day error sums into paired bootstrap confidence intervals on the ablation-table deltas.
- **E6** is the limitations write-up.
- **E7** prints the finished ablation table with CIs attached, and is the Gate E stop.

In [8]:
# ── E2 · Classification baselines: persistence & profile-class ─────────────────
# Two baselines for the classification metric, mirroring the flow HL/HA baselines from cell 06:
# persistence holds the class known at the window's last input step forward across all 12 horizons;
# profile-class is the train-fitted majority class per (sensor, 15-min slot, day-type).
def confusion_from_labels(true, pred, n_classes=3):
    true, pred = true.ravel(), pred.ravel()
    keep = (true >= 0) & (pred >= 0)
    return np.bincount((true[keep].astype(np.int64) * n_classes + pred[keep].astype(np.int64)),
                       minlength=n_classes ** 2).reshape(n_classes, n_classes).astype(np.float64)

PROFILE_CLASS = np.full((3, STEPS_PER_DAY, N), -1, np.int8)
for d in range(3):
    for k in range(STEPS_PER_DAY):
        sel = (dtype3[:TRAIN_END] == d) & (slot[:TRAIN_END] == k)
        Lk = LAB[:TRAIN_END][sel]
        counts = np.stack([(Lk == c).sum(0) for c in range(3)])
        PROFILE_CLASS[d, k] = np.where(counts.sum(0) > 0, counts.argmax(0), -1)

test_starts = idx['test']
base_class = LAB[test_starts]                                     # class at the window's last input step

pers_conf = np.zeros((HORIZON, 3, 3))
prof_conf = np.zeros((HORIZON, 3, 3))
for h in range(HORIZON):
    t = h + 1
    true_h = LAB[test_starts + t]
    pers_conf[h] = confusion_from_labels(true_h, base_class)
    prof_h = PROFILE_CLASS[dtype3[test_starts + t], slot[test_starts + t]]
    prof_conf[h] = confusion_from_labels(true_h, prof_h)

_, pers_rec, _, pers_macro = conglib.class_scores(pers_conf)
_, prof_rec, _, prof_macro = conglib.class_scores(prof_conf)

baselines = pd.DataFrame({'persistence': [pers_macro.mean(), *pers_rec.mean(0)],
                          'profile-class': [prof_macro.mean(), *prof_rec.mean(0)]},
                         index=['macro-F1', 'recall_free', 'recall_heavy', 'recall_congested'])
print('Classification baselines (test set, averaged over all 12 horizons):')
print(baselines.round(3).to_string())

print('\nR1–R6 macro-F1, for comparison (already saved, not recomputed):')
for r in ('R1', 'R2', 'R3', 'R4', 'R5', 'R6'):
    f = json.load(open(DRIVE_PROJ / 'results' / f'{r}_final.json'))['final']['test']['cls']
    print(f'  {r}: macro-F1 {f["macro_f1"]:.3f}  (h1 {f["macro_f1_h"][0]:.3f} -> h12 {f["macro_f1_h"][-1]:.3f})')
save_result('eval_baselines', {'persistence_macro_f1': float(pers_macro.mean()),
                               'profile_class_macro_f1': float(prof_macro.mean()),
                               'persistence_recall': pers_rec.mean(0).tolist(),
                               'profile_class_recall': prof_rec.mean(0).tolist()})

Classification baselines (test set, averaged over all 12 horizons):
                  persistence  profile-class
macro-F1                0.656          0.761
recall_free             0.951          0.969
recall_heavy            0.496          0.656
recall_congested        0.522          0.667

R1–R6 macro-F1, for comparison (already saved, not recomputed):
  R1: macro-F1 0.768  (h1 0.782 -> h12 0.751)
  R2: macro-F1 0.769  (h1 0.785 -> h12 0.752)
  R3: macro-F1 0.780  (h1 0.790 -> h12 0.766)
  R4: macro-F1 0.790  (h1 0.799 -> h12 0.777)
  R5: macro-F1 0.787  (h1 0.799 -> h12 0.774)
  R6: macro-F1 0.787  (h1 0.798 -> h12 0.775)


In [9]:
# ── E3 · Stratification masks: rain, incident, peak, holiday-period, true-class ────────────────
# Every mask is (T, N) bool over the full timeline, so it can be indexed the same way as F/LAB at
# window origin + horizon. All train-independent (these describe test-period conditions, not fitted values).
inc_f = np.load(DRIVE_PROJ / 'factors' / 'incidents.npz')
INCIDENT_MASK = inc_f['active'] > 0

c4 = np.load(DRIVE_PROJ / 'factors' / 'inputs_4c.npz')
RAIN_MASK = (c4['wx'][..., 0].astype(np.float32) @ c4['w_idw'].T) > 0.5    # per-sensor IDW rain flag, matches what R5 sees

b4 = np.load(DRIVE_PROJ / 'factors' / 'inputs_4b.npz')
HOLIDAY_MASK = np.broadcast_to((b4['time'][:, 3] > 0.5)[:, None], (T, N))   # 'holiday_period' column (same for all sensors)

hour = TIMES.hour.to_numpy()
wk = np.asarray(TIMES.dayofweek < 5)
PEAK_MASK = np.broadcast_to((wk & (((hour >= 6) & (hour < 9)) | ((hour >= 15) & (hour < 19))))[:, None], (T, N))

CLASS_MASK = {c: (LAB == i) for i, c in enumerate(['free', 'heavy', 'congested'])}

tst = slice(idx['test'][0] + 1, T)                                 # target-time span of the test set
print('mask coverage, share of test cells (sensor-timesteps):')
for name, msk in [('rain', RAIN_MASK), ('dry', ~RAIN_MASK), ('incident', INCIDENT_MASK),
                  ('non-incident', ~INCIDENT_MASK), ('peak', PEAK_MASK), ('off-peak', ~PEAK_MASK),
                  ('holiday-period', HOLIDAY_MASK), ('non-holiday', ~HOLIDAY_MASK), *CLASS_MASK.items()]:
    print(f'  {name:>14}: {msk[tst].mean():.4f}')
print(f'\nnetwork-average rainy 15-min steps in test: {(RAIN_MASK[tst].mean(1) > 0.5).sum() / 4:.0f} h '
      '(cross-check vs the ~147 h reported in Stage F)')
print(f'incident-affected test cells: {INCIDENT_MASK[tst].mean():.4f} (cross-check vs the ~3% reported in Stage I)')

mask coverage, share of test cells (sensor-timesteps):
            rain: 0.0830
             dry: 0.9170
        incident: 0.0307
    non-incident: 0.9693
            peak: 0.2076
        off-peak: 0.7924
  holiday-period: 0.1232
     non-holiday: 0.8768
            free: 0.7458
           heavy: 0.0621
       congested: 0.0467

network-average rainy 15-min steps in test: 147 h (cross-check vs the ~147 h reported in Stage F)
incident-affected test cells: 0.0307 (cross-check vs the ~3% reported in Stage I)


In [10]:
# ── E4 · One forward pass per finished run: stratified flow-MAE + the bootstrap primitive ──────
MODELS = ['R0', 'A1', 'A2', 'A3', 'R1', 'R2', 'R3', 'R4', 'R5', 'R6']
STRATA = {'rain': RAIN_MASK, 'dry': ~RAIN_MASK, 'incident': INCIDENT_MASK, 'non-incident': ~INCIDENT_MASK,
         'peak': PEAK_MASK, 'off-peak': ~PEAK_MASK, 'holiday-period': HOLIDAY_MASK, 'non-holiday': ~HOLIDAY_MASK,
         **{f'true={c}': m for c, m in CLASS_MASK.items()}}

y_off_np = np.arange(1, HORIZON + 1)
idx_th = test_starts[:, None] + y_off_np[None, :]                 # (S, HORIZON) absolute time index per (window, horizon)

strat_table = {}          # run -> {stratum: MAE}
day_err = {}              # run -> (err_sum_per_window, err_cnt_per_window), summed over horizon+sensor

for run in MODELS:
    t0 = time.time()
    model, data, mean, std, args = conglib.load_run(run, DATA, DRIVE_PROJ, REPO, DEVICE, adj, F, LAB=LAB)
    mt = args['task'] == 'mt'
    if mt:
        pf, tf, _, _ = conglib.predict_mt(model, data, test_starts, mean, std, False, bs=EVAL_BS)
    else:
        pf, tf = conglib.predict(model, data, test_starts, mean, std, False, bs=EVAL_BS)
    pf, tf = pf.cpu().numpy(), tf.cpu().numpy()
    del model, data
    conglib.reset_mem(DEVICE)

    valid = tf != 0
    abs_err = np.abs(pf - tf)
    day_err[run] = ((abs_err * valid).sum(axis=(1, 2)), valid.sum(axis=(1, 2)))

    row = {'overall': (abs_err * valid).sum() / valid.sum()}
    for name, mask in STRATA.items():
        mm = valid & mask[idx_th]
        row[name] = (abs_err * mm).sum() / max(mm.sum(), 1)
    strat_table[run] = row
    print(f'{run:3s} ({"mt " if mt else "flw"}, {time.time() - t0:5.1f}s): overall {row["overall"]:.3f}   '
          + '  '.join(f'{k} {v:.2f}' for k, v in row.items() if k != 'overall'), flush=True)
    del pf, tf, abs_err, valid

strat_df = pd.DataFrame(strat_table).T
print('\nFull stratified test flow-MAE table:')
print(strat_df.round(3).to_string())
save_result('eval_stratified', strat_df.round(4).to_dict())

check supports length 2 3
R0  (flw, 113.2s): overall 20.347   rain 28.17  dry 19.64  incident 30.09  non-incident 20.03  peak 32.07  off-peak 17.27  holiday-period 24.74  non-holiday 19.74  true=free 17.45  true=heavy 30.09  true=congested 44.33
check supports length 2 2
A1  (flw,  83.9s): overall 22.901   rain 29.87  dry 22.27  incident 32.68  non-incident 22.59  peak 36.02  off-peak 19.46  holiday-period 26.62  non-holiday 22.38  true=free 19.94  true=heavy 35.18  true=congested 49.46
check supports length 0 1
A2  (flw,  60.3s): overall 21.841   rain 29.83  dry 21.12  incident 31.83  non-incident 21.52  peak 35.24  off-peak 18.32  holiday-period 26.06  non-holiday 21.25  true=free 18.70  true=heavy 32.74  true=congested 47.88
check supports length 0 0
A3  (flw,  43.3s): overall 27.921   rain 32.42  dry 27.51  incident 38.20  non-incident 27.59  peak 45.57  off-peak 23.28  holiday-period 28.76  non-holiday 27.80  true=free 24.25  true=heavy 46.58  true=congested 61.97
check supports l

In [11]:
# ── E5 · Paired day-block bootstrap on the ablation deltas (uses E4's day_err) ──────────────────
day_id = TIMES[test_starts].normalize().to_numpy()          # calendar day of each window's ORIGIN (a documented
uniq_days = np.unique(day_id)                                # simplification: a 3h horizon can cross midnight)
print(f'{len(uniq_days)} distinct test-origin days for the block bootstrap')

day_pos_of = pd.Series(np.arange(len(uniq_days)), index=uniq_days)
window_day_pos = day_pos_of.loc[day_id].to_numpy()

day_sum, day_cnt = {}, {}
for run, (es, ec) in day_err.items():
    ds, dc = np.zeros(len(uniq_days)), np.zeros(len(uniq_days))
    np.add.at(ds, window_day_pos, es)
    np.add.at(dc, window_day_pos, ec)
    day_sum[run], day_cnt[run] = ds, dc

rng = np.random.default_rng(SEED)
N_BOOT = 1000
boot_pos = rng.integers(0, len(uniq_days), size=(N_BOOT, len(uniq_days)))   # same resampled days for every
boot = {run: day_sum[run][boot_pos].sum(1) / day_cnt[run][boot_pos].sum(1)  # model -> paired comparisons
       for run in day_sum}

DELTAS = [('R1', 'R2'), ('R2', 'R3'), ('R3', 'R4'), ('R4', 'R5'), ('R4', 'R6'), ('A3', 'A1'), ('A3', 'A2'), ('A3', 'R0')]
rows = []
for a_run, b_run in DELTAS:
    point = day_sum[b_run].sum() / day_cnt[b_run].sum() - day_sum[a_run].sum() / day_cnt[a_run].sum()
    diff = boot[b_run] - boot[a_run]
    lo, hi = np.percentile(diff, [2.5, 97.5])
    rows.append({'from': a_run, 'to': b_run, 'delta_MAE': point, 'CI_lo': lo, 'CI_hi': hi,
                'significant': bool((lo > 0) or (hi < 0))})
boot_df = pd.DataFrame(rows)
print('Paired day-block bootstrap (1000 resamples of test days), delta = to − from (negative = improvement):')
print(boot_df.round(3).to_string(index=False))
save_result('eval_bootstrap', boot_df.to_dict('records'))

74 distinct test-origin days for the block bootstrap
Paired day-block bootstrap (1000 resamples of test days), delta = to − from (negative = improvement):
from to  delta_MAE  CI_lo  CI_hi  significant
  R1 R2     -0.018 -0.140  0.135        False
  R2 R3     -1.486 -1.811 -1.233         True
  R3 R4     -0.346 -0.604 -0.131         True
  R4 R5      0.080 -0.082  0.264        False
  R4 R6      0.144 -0.029  0.343        False
  A3 A1     -5.020 -5.732 -4.345         True
  A3 A2     -6.079 -6.939 -5.274         True
  A3 R0     -7.574 -8.488 -6.680         True


## Limitations

- **Congestion is measured, not observed directly.** LargeST gives flow only; the free/heavy/congested label comes from PeMS occupancy, which is itself an inferred signal (per-lane loop-detector on-time), not a direct queue measurement. The occupancy thresholds are fitted per sensor and validated (bottleneck locations, holiday drop, incident lift), but they are still a modelling choice, not ground truth.
- **PeMS occupancy is itself partly imputed.** About 10% of sensor-hours in the underlying PeMS data are below 50% observed and were excluded from label fitting, but the flow series they feed (via LargeST) was not separately filtered — the flow numbers may include PeMS's own imputation in those hours.
- **The 999 flow ceiling looks like sensor/field saturation on a handful of stations**, not a modelling artefact — confirmed against PeMS's own 5-minute archive for the same sensors, which shows the same values are genuinely reported that high, not truncated by our pipeline.
- **Lane count for one sensor (ID 1108333) was corrected by hand** (metadata said 2 lanes; its observed flow implies ~5) — a one-off fix based on a plausibility check, not a systematic audit of all 716 sensors' metadata.
- **Test-only holidays.** Thanksgiving and the Dec 21–31 period fall only in the test split (never in train or validation), so the model has to generalise holiday behaviour from smaller analogues (New Year's, Memorial Day, July 4th, etc.) it saw in training. R4's gain from the holiday/school features is real on this split, but the biggest test-period anomalies (Thanksgiving week, Christmas) are exactly the days the model had the least similar training exposure to.
- **Single seed, fixed epoch budget.** Every run used seed 2023 and 16 epochs (warmup → constant LR → cosine decay) on a laptop GPU, versus the paper's much longer schedule (~80 epochs on a data-centre GPU). Row-to-row deltas smaller than typical seed-to-seed noise should be read with that in mind — the day-block bootstrap CIs above quantify the *test-sampling* uncertainty, but not the *training-seed* uncertainty, which was not measured (one seed only, given the compute budget).
- **MPS (Apple GPU) numeric limitations.** Training and evaluation ran in fp32 throughout (mixed precision measured ~11% faster on MPS but wasn't worth the added complexity across a laptop-only budget); the day-block bootstrap and stratified metrics above were computed on CPU/MPS in float64 after inference, so this doesn't affect their precision.
- **Day-block bootstrap blocks by the forecast's origin day**, not by each horizon's own target day — a documented simplification, since a 3-hour-ahead forecast made at 23:00 targets the next calendar day. This is the standard reading of "day-block" for a rolling-origin forecast and doesn't materially change which days are grouped together for all but a small fraction of late-evening windows.
- **Weather and incident coverage are limited by what's actually in the test period.** The test split has only ~147 hours of measurable network rain and incident-affected cells are ~3% of the test set — real effects, but on a comparatively small slice of the data, so the R5/R6 stratified numbers above should be read as suggestive rather than as tightly bounded estimates.

## ⛔ GATE E — STOP

E4/E5's output above is the final scientific record for the writeup: classification baselines (E2), the stratified flow-MAE table (E4), and the bootstrap CIs on every ablation-table delta (E5). Before this feeds the dashboard/chatbot/deck, check:
- does R4 → R5 (weather) actually improve inside the **rain** stratum, even though the aggregate barely moved?
- does R4 → R6 (incidents) improve inside the **incident** stratum?
- which deltas in the final table have a 95% CI that **excludes zero** (a defensible "this row helped" claim) versus one that straddles zero (not distinguishable from noise at this sample size)?

Once you've reviewed this, tell me to proceed and I'll move to the prediction export + dashboard/chatbot work.

---

### Gate E — reviewed 2026-09-18

Bootstrap deltas (1000 resamples, 74 test-origin days), delta = to − from:

| from → to | ΔMAE | 95% CI | significant |
|---|---|---|---|
| R1 → R2 | −0.018 | [−0.140, 0.135] | no |
| R2 → R3 | −1.486 | [−1.811, −1.233] | **yes** |
| R3 → R4 | −0.346 | [−0.604, −0.131] | **yes** |
| R4 → R5 | +0.080 | [−0.082, 0.264] | no |
| R4 → R6 | +0.144 | [−0.029, 0.343] | no |
| A3 → A1 | −5.020 | [−5.732, −4.345] | **yes** |
| A3 → A2 | −6.079 | [−6.939, −5.274] | **yes** |
| A3 → R0 | −7.574 | [−8.488, −6.680] | **yes** |

**R4 is the statistically-best model** — R2→R3 (4a features) and R3→R4 (4b holiday/calendar) are both real, CI-excludes-zero improvements. R4→R5 and R4→R6 are not distinguishable from noise in aggregate.

Follow-up stratum checks (point estimates, no separate CI computed):
- **R4 → R5 inside the rain stratum**: 26.153 → 25.791 (**−0.36, improves**) — weather has a real, localized effect masked at the aggregate level because rain is only 8.3% of test cells.
- **R4 → R6 inside the incident stratum**: 28.400 → 28.530 (**+0.13, worse**) — the incident feature shows no localized benefit even where incidents are active; unlike weather, this isn't just aggregate dilution.

Decision: ship **R4** as the model behind the dashboard/chatbot/deck. Note the R5 rain finding as a genuine (if minor) positive result worth a line in the writeup; note the R6 finding as a negative result (incident features didn't help, even locally) rather than omitting it.

Proceeding to prediction export + dashboard/chatbot.
